In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier


In [3]:
df = pd.read_csv('results.csv')
result_df_1 = df.copy()
result_df_1.dropna(inplace= True)
print(result_df_1.shape)
result_df_1.tail()

(49215, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49210,2026-03-31,Kosovo,Turkey,0.0,1.0,FIFA World Cup qualification,Pristina,Kosovo,False
49211,2026-03-31,Czech Republic,Denmark,2.0,2.0,FIFA World Cup qualification,Prague,Czech Republic,False
49212,2026-03-31,Cameroon,China PR,2.0,0.0,FIFA Series,Melbourne,Australia,True
49213,2026-03-31,Australia,Curaçao,5.0,1.0,FIFA Series,Melbourne,Australia,False
49214,2026-03-31,Kazakhstan,Comoros,1.0,0.0,FIFA Series,Astana,Kazakhstan,False


In [4]:
result_df_1.isna().sum()

date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

In [5]:
"Turkiye" in result_df_1.values

False

In [6]:
print("Is Yugoslavia a Home Team?", "Yugoslavia" in result_df_1["home_team"].values)
print("Is Yugoslavia an Away Team?", "Yugoslavia" in result_df_1["away_team"].values)

Is Yugoslavia a Home Team? True
Is Yugoslavia an Away Team? True


In [7]:
change_team = {
    'Soviet Union' : 'Russia', 
    'Yugoslavia' : 'Serbia',
    'German DR' : 'East Germany',
    'Czech Republic': 'Czechia',
    'DR Congo' : 'Congo',
    'China PR' : 'China',
    'Republic of Ireland' : 'Ireland',
    "American Samoa": "Eastern Samoa",
    "Iraqi Kurdistan": "Kurdistan",
    "Macau": "Macao",
    "Micronesia": "Federated States of Micronesia",
    "Réunion": "Reunion",
    "Saint Barthélemy": "Saint Barthelemy",
    "São Tomé and Príncipe": "Sao Tome and Principe",
    "Timor-Leste": "East Timor",
    "Vatican City": "Vatican",
    "Vietnam Republic": "South Vietnam",
    "Wallis Islands and Futuna": "Wallis and Futuna"
}

In [8]:
result_df_1["home_team"] = result_df_1["home_team"].replace(change_team)
result_df_1["away_team"] = result_df_1["away_team"].replace(change_team)
result_df_1["country"] = result_df_1["country"].replace(change_team)


In [9]:
"Yugoslavia" in result_df_1.values
print("Is Yugoslavia a Home Team?", "Yugoslavia" in result_df_1["home_team"].values)
print("Is Yugoslavia an Away Team?", "Yugoslavia" in result_df_1["away_team"].values)
print("Is Yugoslavia a Host Country?", "Yugoslavia" in result_df_1["country"].values)

Is Yugoslavia a Home Team? False
Is Yugoslavia an Away Team? False
Is Yugoslavia a Host Country? False


In [10]:
"German DR" in result_df_1.values

False

In [11]:
teams = ['Australia','Iran', 'Iraq', 'Japan', 'Jordan', 'Qatar', 'Saudi Arabia',
          'South Korea', 'Uzbekistan', 'Algeria', 'Cape Verde', 'Congo', "Ivory Coast",
            'Egypt', 'Ghana', 'Morocco', 'Senegal', 'South Africa','Tunisia', 'Canada',
              'Mexico', 'United States', 'Curaçao', 'Haiti', 'Panama', 'Argentina', 
              'Brazil', 'Colombia', 'Ecuador', 'Paraguay', 'Uruguay', 'New Zealand',
                'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Croatia', 'Czechia'
                  ,'England', 'France' , 'Germany', 'Netherlands', 'Norway',
                    'Portugal', 'Scotland', 'Spain', 'Sweden', 'Switzerland', 'Turkey']
                    

In [12]:
# result_df = result_df_1[(result_df_1['home_team'].isin(teams)) | (result_df_1['away_team'].isin(teams))]

In [13]:
# result_df

In [14]:
# result_df.dropna(axis = 0, inplace= True)

In [15]:
# result_df.isna().sum()

In [16]:
result_df_1['home_score'] = result_df_1['home_score'].astype(int)
result_df_1['away_score'] = result_df_1['away_score'].astype(int)

In [17]:
elo_df_1 = pd.read_csv('eloratings.csv')
print(elo_df_1.shape)
elo_df_1

(6678, 4)


,date,team,rating,change
0,1872-11-30,England,2003.0,3
1,1872-11-30,Scotland,1997.0,-3
2,1873-03-08,England,2014.0,11
3,1873-03-08,Scotland,1986.0,-11
4,1874-03-07,England,2006.0,-8
...,...,...,...,...
6673,12/13/2025,Northern Mariana Islands,432.0,0
6674,12/13/2025,Cocos Islands,422.0,0
6675,12/13/2025,Palau,402.0,0
6676,12/13/2025,Eastern Samoa,389.0,0


In [18]:
elo_df_1[elo_df_1.isna().any(axis=1)]
elo_df_1.isna().sum()

date       0
team       0
rating    31
change     0
dtype: int64

In [19]:
elo_df_1[elo_df_1['team'] == 'Moldova']


,date,team,rating,change
1219,10/14/1992,Moldova,NaN,-4
1346,9/6/1995,Moldova,NaN,-7
1536,10/7/1997,Moldova,NaN,-19
1775,9/1/2001,Moldova,NaN,15
1824,2/13/2002,Moldova,NaN,-32
1891,8/21/2002,Moldova,NaN,-6
2137,9/4/2004,Moldova,NaN,-10
2234,6/4/2005,Moldova,NaN,-10
2380,9/7/2005,Moldova,NaN,-15
2706,10/17/2007,Moldova,NaN,15


In [20]:
moldova_df = elo_df_1[elo_df_1['team'] == 'Moldova'].sort_values('date').copy()
known_2026_rating = 1288

total_historical_change = moldova_df['change'].iloc[1:].sum()
initial_rating = known_2026_rating - total_historical_change
rolling_changes = moldova_df['change'].copy()
rolling_changes.iloc[0] = 0

moldova_df['rating'] = initial_rating + rolling_changes.cumsum()

elo_df_1.update(moldova_df)
elo_df_1[elo_df_1['team'] == 'Moldova']





,date,team,rating,change
1219,10/14/1992,Moldova,1352.0,-4
1346,9/6/1995,Moldova,1326.0,-7
1536,10/7/1997,Moldova,1335.0,-19
1775,9/1/2001,Moldova,1351.0,15
1824,2/13/2002,Moldova,1344.0,-32
1891,8/21/2002,Moldova,1336.0,-6
2137,9/4/2004,Moldova,1335.0,-10
2234,6/4/2005,Moldova,1345.0,-10
2380,9/7/2005,Moldova,1290.0,-15
2706,10/17/2007,Moldova,1354.0,15


In [21]:
elo_df_1.isna().sum()

date      0
team      0
rating    0
change    0
dtype: int64

In [22]:
elo_df_1["date"] = pd.to_datetime(elo_df_1["date"], format= 'mixed')
elo_df_1 = elo_df_1.sort_values(by="date")

latest_elo_df = elo_df_1.groupby("team").last().reset_index()
latest_elo_df = latest_elo_df[["team", "date", "rating"]]
latest_elo_df = latest_elo_df.sort_values(by="rating", ascending=False)

# print(latest_elo_df)

# latest_elo_df.to_csv("latest_team_elo.csv", index=False)

latest_elo_df




,team,date,rating
224,Spain,2025-12-13,2171.0
264,West Germany,1974-09-04,2144.0
7,Argentina,2025-12-13,2113.0
83,France,2025-12-13,2062.0
72,England,2025-12-13,2042.0
...,...,...,...
165,Niue,2025-12-13,496.0
168,Northern Mariana Islands,2025-12-13,432.0
47,Cocos Islands,2025-12-13,422.0
177,Palau,2025-12-13,402.0


In [23]:
latest_elo_df['team'] = latest_elo_df['team'].str.strip()

In [24]:
change_team = {
    'Soviet Union' : 'Russia',
    'Yugoslavia' : 'Serbia',
    'West Germany' : 'Germany',
    'Macedonia' : 'North Macedonia',
    'US Virgin Islands' : 'United States Virgin Islands',
    
}

elo_df_1['team'] = elo_df_1['team'].replace(r'\s+', ' ', regex=True).str.strip()

elo_df_1['team'] = elo_df_1['team'].replace(change_team)

print("Did it work for Germany?", "Germany" in elo_df_1['team'].values)
print("Is West Germany gone?", "West Germany" in elo_df_1['team'].values)

Did it work for Germany? True
Is West Germany gone? False


In [25]:
"West Germany" in latest_elo_df.values

False

In [26]:
# Grabs the name from row index 264
test_name = latest_elo_df.loc[264, 'team']
print(f"-->{test_name}<--")

-->West Germany<--


In [27]:
elo_df_1

,date,team,rating,change
0,1872-11-30,England,2003.0,3
1,1872-11-30,Scotland,1997.0,-3
2,1873-03-08,England,2014.0,11
3,1873-03-08,Scotland,1986.0,-11
4,1874-03-07,England,2006.0,-8
...,...,...,...,...
6519,2025-12-13,Palestine,1470.0,0
6520,2025-12-13,Curaçao,1467.0,0
6521,2025-12-13,Gabon,1467.0,0
6523,2025-12-13,Bulgaria,1452.0,0


In [28]:
matches_teams = set(result_df_1['home_team'].unique()).union(set(result_df_1['away_team'].unique()))
elo_teams = set(elo_df_1['team'].unique())


In [29]:
missing_elo = sorted(list(matches_teams - elo_teams))
print(len(missing_elo))
missing_elo

87


['Abkhazia',
 'Alderney',
 'Ambazonia',
 'Andalusia',
 'Arameans Suryoye',
 'Artsakh',
 'Asturias',
 'Aymara',
 'Barawa',
 'Basque Country',
 'Biafra',
 'Brittany',
 'Canary Islands',
 'Cascadia',
 'Catalonia',
 'Central Spain',
 'Chameria',
 'Chechnya',
 'Cilento',
 'Corsica',
 'County of Nice',
 'Crimea',
 'Darfur',
 'Donetsk PR',
 'Délvidék',
 'Elba Island',
 'Ellan Vannin',
 'Felvidék',
 'Franconia',
 'Frøya',
 'Galicia',
 'Gotland',
 'Gozo',
 'Guernsey',
 'Găgăuzia',
 'Hitra',
 'Hmong',
 'Isle of Man',
 'Isle of Wight',
 'Jersey',
 'Kabylia',
 'Kernow',
 'Kárpátalja',
 'Luhansk PR',
 'Madrid',
 'Manchukuo',
 'Mapuche',
 'Matabeleland',
 'Maule Sur',
 'Menorca',
 'Occitania',
 'Orkney',
 'Padania',
 'Panjab',
 'Parishes of Jersey',
 'Provence',
 'Raetia',
 'Republic of St. Pauli',
 'Rhodes',
 'Romani people',
 'Ryūkyū',
 'Saare County',
 'Saarland',
 'Saint Helena',
 'Sark',
 'Saugeais',
 'Sealand',
 'Seborga',
 'Shetland',
 'Silesia',
 'South Ossetia',
 'Surrey',
 'Székely Land',


In [30]:
missing_matches = sorted(list(elo_teams - matches_teams))
print(len(missing_matches))
missing_matches


22


['British Guiana',
 'Burma',
 'Ceylon',
 'Christmas Island',
 'Cocos Islands',
 'Congo-Brazzaville',
 'Dahomey',
 'Democratic Republic of Congo',
 'Khmer Republic',
 'Malaya',
 'Netherlands Antilles',
 'North Yemen',
 'Northern Rhodesia',
 'Saba',
 'Serbia and Montenegro',
 'Sint Eustatius',
 'Southern Rhodesia',
 'Surinam',
 'Swaziland',
 'Tanganyika',
 'United Arab Republic',
 'Upper Volta']

In [31]:
result_df_1.tail(30)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49185,2026-03-31,Jordan,Nigeria,2,2,Friendly,Antalya,Turkey,True
49186,2026-03-31,Liberia,Libya,2,2,"Morocco, Capital of African Football",Casablanca,Morocco,True
49187,2026-03-31,Botswana,Malawi,1,0,Mukuru 4 Nations,Francistown,Botswana,False
49188,2026-03-31,Mexico,Belgium,1,1,Friendly,Chicago,United States,True
49189,2026-03-31,Montenegro,Slovenia,2,3,Friendly,Podgorica,Montenegro,False
49190,2026-03-31,Morocco,Paraguay,2,1,Friendly,Lens,France,True
49191,2026-03-31,Netherlands,Ecuador,1,1,Friendly,Eindhoven,Netherlands,False
49192,2026-03-31,Niger,Togo,0,1,"Morocco, Capital of African Football",Casablanca,Morocco,True
49193,2026-03-31,Norway,Switzerland,0,0,Friendly,Oslo,Norway,False
49194,2026-03-31,Peru,Honduras,2,2,Friendly,Madrid,Spain,True


In [32]:


result_df_1['home_team'] = result_df_1['home_team'].str.strip()
result_df_1['away_team'] = result_df_1['away_team'].str.strip()
elo_df_1['team'] = elo_df_1['team'].str.strip()

result_df_1['date'] = pd.to_datetime(result_df_1['date'])
elo_df_1['date'] = pd.to_datetime(elo_df_1['date'])

result_df = result_df_1.sort_values('date')
elo_df_1 = elo_df_1.sort_values('date')



In [33]:
merged_df = pd.merge_asof(
    result_df, 
    elo_df_1, 
    left_on='date', 
    right_on='date', 
    left_by='home_team', 
    right_by='team', 
    direction='backward'
)

merged_df = merged_df.rename(columns={'rating': 'home_team_elo'})

merged_df = pd.merge_asof(
    merged_df, 
    elo_df_1, 
    left_on='date', 
    right_on='date', 
    left_by='away_team', 
    right_by='team', 
    direction='backward'
)

merged_df = merged_df.rename(columns={'rating': 'away_team_elo'})

merged_df = merged_df.drop(columns=['team_x', 'team_y'], errors='ignore')

In [34]:
print(merged_df.shape)
merged_df

(49215, 13)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_team_elo,change_x,away_team_elo,change_y
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,1997.0,-3.0,2003.0,3.0
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,2014.0,11.0,1986.0,-11.0
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,1994.0,8.0,2006.0,-8.0
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,2003.0,-3.0,1997.0,3.0
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,2010.0,13.0,1990.0,-13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
49210,2026-03-31,Mexico,Belgium,1,1,Friendly,Chicago,United States,True,1835.0,0.0,1849.0,0.0
49211,2026-03-31,Montenegro,Slovenia,2,3,Friendly,Podgorica,Montenegro,False,1444.0,0.0,1695.0,0.0
49212,2026-03-31,Morocco,Paraguay,2,1,Friendly,Lens,France,True,1830.0,0.0,1833.0,0.0
49213,2026-03-31,Niger,Togo,0,1,"Morocco, Capital of African Football",Casablanca,Morocco,True,1403.0,0.0,1340.0,0.0


In [35]:
merged_df.isna().sum()

date                0
home_team           0
away_team           0
home_score          0
away_score          0
tournament          0
city                0
country             0
neutral             0
home_team_elo    4442
change_x         4442
away_team_elo    4484
change_y         4484
dtype: int64

In [36]:
player_df = pd.read_csv('players.csv')
print(player_df.shape)
player_df

(47689, 26)


,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus Football Club,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,IFM,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,FC Bayern München,100000.0,3000000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47684,1529719,Eun-seok,Ko,Eun-seok Ko,2025,21459,eun-seok-ko,NaN,NaN,"Korea, South",...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/eun-seok-ko/pr...,RSK1,Gangwon Football Club,NaN,NaN
47685,1535382,Wojciech,Pańkowski,Wojciech Pańkowski,2025,6456,wojciech-pankowski,NaN,NaN,Poland,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/wojciech-panko...,PL1,GKS GieKSa Katowice Spółka Akcyjna,NaN,NaN
47686,1545440,Borys,Mołdach,Borys Mołdach,2025,3527,borys-moldach,NaN,NaN,Poland,...,ProSport Manager,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/borys-moldach/...,PL1,Motor Lublin Spółka Akcyjna,NaN,NaN
47687,1545442,Kacper,Surowiec,Kacper Surowiec,2025,15906,kacper-surowiec,NaN,NaN,Poland,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/kacper-surowie...,PL1,Termalica Bruk-Bet Nieciecza Klub Sportowy,NaN,NaN


In [37]:
player_df.isna().sum()

player_id                                   0
first_name                               3092
last_name                                   0
name                                        0
last_season                                 0
current_club_id                             0
player_code                                 0
country_of_birth                         5166
city_of_birth                            4890
country_of_citizenship                    273
date_of_birth                              49
sub_position                              465
position                                    0
foot                                     5100
height_in_cm                             3659
contract_expiration_date                16376
agent_name                              22068
image_url                                   0
international_caps                      30127
international_goals                     30127
current_national_team_id                45231
url                               

In [38]:
modern_df = merged_df[merged_df['date'].dt.year >= 2004].copy()
print(modern_df.shape)
# modern_df.to_csv('moderndf.csv')

(21371, 13)


In [39]:
modern_df.isna().sum()

date               0
home_team          0
away_team          0
home_score         0
away_score         0
tournament         0
city               0
country            0
neutral            0
home_team_elo    870
change_x         870
away_team_elo    888
change_y         888
dtype: int64

In [40]:
modern_df[modern_df.isna().any(axis=1)]

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_team_elo,change_x,away_team_elo,change_y
27950,2004-02-18,Vietnam,Maldives,4,0,FIFA World Cup qualification,Hanoi,Vietnam,False,1252.0,-3.0,NaN,NaN
27957,2004-02-18,Kyrgyzstan,Tajikistan,1,2,FIFA World Cup qualification,Bishkek,Kyrgyzstan,False,1168.0,-12.0,NaN,NaN
27966,2004-02-18,Haiti,Turks and Caicos Islands,5,0,FIFA World Cup qualification,Miami,United States,True,1434.0,4.0,NaN,NaN
27981,2004-02-21,Turks and Caicos Islands,Haiti,0,2,FIFA World Cup qualification,Hialeah,United States,True,NaN,NaN,1434.0,4.0
27995,2004-03-06,Alderney,Jersey,0,4,Muratti Vase,Alderney,Alderney,False,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
48475,2025-07-17,Gozo,Western Isles,3,3,Island Games,Dounby,Scotland,True,NaN,NaN,NaN,NaN
48476,2025-07-17,Isle of Man,Ynys Môn,0,1,Island Games,Kirkwall,Scotland,True,NaN,NaN,NaN,NaN
48477,2025-07-18,Isle of Man,Jersey,3,2,Island Games,Kirkwall,Scotland,True,NaN,NaN,NaN,NaN
48479,2025-08-14,Marshall Islands,United States Virgin Islands,0,4,Outrigger Challenge Cup,United States,United States,True,NaN,NaN,586.0,-15.0


In [41]:
modern_df = modern_df.dropna(subset=['home_team_elo', 'away_team_elo'], how='all')

In [42]:
baseline_elo = 800

modern_df['home_team_elo'] = modern_df['home_team_elo'].fillna(baseline_elo)
modern_df['away_team_elo'] = modern_df['away_team_elo'].fillna(baseline_elo)


In [43]:
modern_df.isna().sum()

date               0
home_team          0
away_team          0
home_score         0
away_score         0
tournament         0
city               0
country            0
neutral            0
home_team_elo      0
change_x         256
away_team_elo      0
change_y         274
dtype: int64

In [44]:
modern_df = modern_df.drop(columns = ['change_x','change_y','city', 'country'])

In [45]:
modern_df

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo
27844,2004-01-01,Bahrain,Saudi Arabia,0,1,Gulf Cup,True,1479.0,1700.0
27845,2004-01-01,Bermuda,Barbados,0,4,Friendly,False,1305.0,1339.0
27846,2004-01-01,Kuwait,Yemen,4,0,Gulf Cup,False,1623.0,1206.0
27847,2004-01-03,Oman,Bahrain,0,1,Gulf Cup,True,1252.0,1479.0
27848,2004-01-03,United Arab Emirates,Qatar,0,0,Gulf Cup,True,1609.0,1532.0
...,...,...,...,...,...,...,...,...,...
49210,2026-03-31,Mexico,Belgium,1,1,Friendly,True,1835.0,1849.0
49211,2026-03-31,Montenegro,Slovenia,2,3,Friendly,False,1444.0,1695.0
49212,2026-03-31,Morocco,Paraguay,2,1,Friendly,True,1830.0,1833.0
49213,2026-03-31,Niger,Togo,0,1,"Morocco, Capital of African Football",True,1403.0,1340.0


In [46]:
modern_df['neutral'] = modern_df['neutral'].astype(int)

In [47]:
modern_df

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo
27844,2004-01-01,Bahrain,Saudi Arabia,0,1,Gulf Cup,1,1479.0,1700.0
27845,2004-01-01,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0
27846,2004-01-01,Kuwait,Yemen,4,0,Gulf Cup,0,1623.0,1206.0
27847,2004-01-03,Oman,Bahrain,0,1,Gulf Cup,1,1252.0,1479.0
27848,2004-01-03,United Arab Emirates,Qatar,0,0,Gulf Cup,1,1609.0,1532.0
...,...,...,...,...,...,...,...,...,...
49210,2026-03-31,Mexico,Belgium,1,1,Friendly,1,1835.0,1849.0
49211,2026-03-31,Montenegro,Slovenia,2,3,Friendly,0,1444.0,1695.0
49212,2026-03-31,Morocco,Paraguay,2,1,Friendly,1,1830.0,1833.0
49213,2026-03-31,Niger,Togo,0,1,"Morocco, Capital of African Football",1,1403.0,1340.0


In [48]:
df1 = pd.read_csv('transfermarkt_national_teams2.csv')
df2 = pd.read_csv('transfermarkt_national_teams3.csv')
df3 = pd.read_csv('transfermarkt_squad_details.csv')
df4 = pd.read_csv('transfermarkt_squad_details2.csv')

team_df = pd.concat([df1,df2], ignore_index= True)
team_df = pd.concat([team_df, df3], ignore_index= True)
team_df = pd.concat([team_df, df4], ignore_index= True)

In [49]:
team_df.to_csv('teamd1.csv')
team_df

,Team,Season,Position,ø-Age,Market value,ø-Market value
0,frankreich,2004,Total:,27.8,€466.65m,€9.93m
1,frankreich,2005,Total:,27.3,€333.75m,€10.43m
2,frankreich,2006,Total:,27.3,€350.90m,€10.63m
3,frankreich,2007,Total:,26.4,€357.20m,€9.92m
4,frankreich,2008,Total:,27.9,€544.00m,€13.60m
...,...,...,...,...,...,...
2777,nordmazedonien,2014,Total:,26.5,€29.93m,€565k
2778,oman,2025,Total:,26.9,€12.58m,€257k
2779,thailand,2012,Total:,28.4,€1.10m,€28k
2780,trinidad-und-tobago,2022,Total:,26.5,€10.73m,€219k


In [50]:
unique_teams = team_df['Team'].unique()
unique_teams

<ArrowStringArray>
[ 'frankreich',     'spanien', 'argentinien',     'england',    'portugal',
   'brasilien', 'niederlande',     'marokko',     'belgien', 'deutschland',
 ...
    'armenien', 'kirgisistan',     'libanon',     'komoren',  'kasachstan',
       'kenia',      'libyen',    'tansania',       'niger', 'mauretanien']
Length: 115, dtype: str

In [51]:
expected_years = range(team_df['Season'].min(), team_df['Season'].max())
expected_years

range(2004, 2026)

In [52]:
full_index = pd.MultiIndex.from_product([unique_teams, expected_years], names=['Team', 'Season'])
actual_index = pd.MultiIndex.from_frame(team_df[['Team', 'Season']])
missing_combinations = full_index.difference(actual_index)
missing_combinations

MultiIndex([], names=['Team', 'Season'])

In [53]:
missing_combinations =pd.DataFrame(index=missing_combinations).reset_index()

In [54]:
missing_combinations.to_csv('missing_years3.csv')

In [55]:
unique_teams = pd.DataFrame(unique_teams)

In [56]:
unique_teams.to_csv('to_be_fixed.csv')

In [57]:
unique_teams_2 = modern_df[['home_team', 'away_team']].stack().unique()

In [58]:
unique_teams_2 = pd.DataFrame(unique_teams_2)
unique_teams_2.to_csv('fixed_answers.csv')

In [59]:
country_mapping = {
    "frankreich": "France",
    "spanien": "Spain",
    "argentinien": "Argentina",
    "england": "England",
    "portugal": "Portugal",
    "brasilien": "Brazil",
    "niederlande": "Netherlands",
    "marokko": "Morocco",
    "belgien": "Belgium",
    "deutschland": "Germany",
    "kroatien": "Croatia",
    "italien": "Italy",
    "kolumbien": "Colombia",
    "senegal": "Senegal",
    "mexiko": "Mexico",
    "vereinigte-staaten": "United States",
    "uruguay": "Uruguay",
    "japan": "Japan",
    "schweiz": "Switzerland",
    "danemark": "Denmark",
    "iran": "Iran",
    "turkei": "Turkey",
    "ecuador": "Ecuador",
    "osterreich": "Austria",
    "sudkorea": "South Korea",
    "nigeria": "Nigeria",
    "australien": "Australia",
    "algerien": "Algeria",
    "agypten": "Egypt",
    "kanada": "Canada",
    "norwegen": "Norway",
    "ukraine": "Ukraine",
    "panama": "Panama",
    "elfenbeinkuste": "Ivory Coast",
    "polen": "Poland",
    "russland": "Russia",
    "wales": "Wales",
    "schweden": "Sweden",
    "serbien": "Serbia",
    "paraguay": "Paraguay",
    "tschechien": "Czechia",
    "ungarn": "Hungary",
    "schottland": "Scotland",
    "tunesien": "Tunisia",
    "kamerun": "Cameroon",
    "demokratische-republik-kongo": "Congo",
    "griechenland": "Greece",
    "slowakei": "Slovakia",
    "venezuela": "Venezuela",
    "usbekistan": "Uzbekistan",
    "costa-rica": "Costa Rica",
    "mali": "Mali",
    "peru": "Peru",
    "chile": "Chile",
    "katar": "Qatar",
    "rumanien": "Romania",
    "irak": "Iraq",
    "slowenien": "Slovenia",
    "irland": "Ireland",
    "sudafrika": "South Africa",
    "saudi-arabien": "Saudi Arabia",
    "burkina-faso": "Burkina Faso",
    "jordanien": "Jordan",
    "albanien": "Albania",
    "bosnien-herzegowina": "Bosnia and Herzegovina",
    "honduras": "Honduras",
    "nordmazedonien": "North Macedonia",
    "vereinigte-arabische-emirate": "United Arab Emirates",
    "kap-verde": "Cape Verde",
    "nordirland": "Northern Ireland",
    "jamaika": "Jamaica",
    "georgien": "Georgia",
    "finnland": "Finland",
    "ghana": "Ghana",
    "island": "Iceland",
    "bolivien": "Bolivia",
    "israel": "Israel",
    "kosovo": "Kosovo",
    "oman": "Oman",
    "guinea": "Guinea",
    "montenegro": "Montenegro",
    "curacao": "Curaçao",
    "haiti": "Haiti",
    "syrien": "Syria",
    "neuseeland": "New Zealand",
    "bulgarien": "Bulgaria",
    "gabun": "Gabon",
    "uganda": "Uganda",
    "angola": "Angola",
    "benin": "Benin",
    "bahrain": "Bahrain",
    "sambia": "Zambia",
    "thailand": "Thailand",
    "china": "China",
    "palastina": "Palestine",
    "guatemala": "Guatemala",
    "belarus": "Belarus",
    "luxemburg": "Luxembourg",
    "vietnam": "Vietnam",
    "el-salvador": "El Salvador",
    "mosambik": "Mozambique",
    "trinidad-und-tobago": "Trinidad and Tobago",
    "tadschikistan": "Tajikistan",
    "madagaskar": "Madagascar",
    "aquatorialguinea": "Equatorial Guinea",
    "armenien": "Armenia",
    "kirgisistan": "Kyrgyzstan",
    "libanon": "Lebanon",
    "komoren": "Comoros",
    "kasachstan": "Kazakhstan",
    "kenia": "Kenya",
    "libyen": "Libya",
    "tansania": "Tanzania",
    "niger": "Niger",
    "mauretanien": "Mauritania"
}

In [60]:
team_df['Team'] = team_df['Team'].replace(country_mapping)

In [61]:
team_df[team_df.isna().any(axis = 1)]

,Team,Season,Position,ø-Age,Market value,ø-Market value
614,Australia,2020,Total:,NaN,-,-
873,Serbia,2004,Total:,NaN,-,-
874,Serbia,2005,Total:,NaN,-,-
1037,Congo,2007,Total:,NaN,-,-
1174,Mali,2006,Total:,NaN,-,-
...,...,...,...,...,...,...
2691,New Zealand,2020,Total:,NaN,-,-
2696,Niger,2009,Total:,NaN,-,-
2738,Trinidad and Tobago,2020,Total:,NaN,-,-
2749,Vietnam,2020,Total:,NaN,-,-


In [62]:
team_df.isna().sum()

Team               0
Season             0
Position           0
ø-Age             76
Market value       0
ø-Market value     0
dtype: int64

In [63]:
# team_df.drop('Position', axis= 1, inplace= True)
team_df

,Team,Season,Position,ø-Age,Market value,ø-Market value
0,France,2004,Total:,27.8,€466.65m,€9.93m
1,France,2005,Total:,27.3,€333.75m,€10.43m
2,France,2006,Total:,27.3,€350.90m,€10.63m
3,France,2007,Total:,26.4,€357.20m,€9.92m
4,France,2008,Total:,27.9,€544.00m,€13.60m
...,...,...,...,...,...,...
2777,North Macedonia,2014,Total:,26.5,€29.93m,€565k
2778,Oman,2025,Total:,26.9,€12.58m,€257k
2779,Thailand,2012,Total:,28.4,€1.10m,€28k
2780,Trinidad and Tobago,2022,Total:,26.5,€10.73m,€219k


In [64]:
team_df.loc[614, 'ø-Age'] = 27.7
team_df.loc[873, 'ø-Age'] = 25.5
team_df.loc[874, 'ø-Age'] = 25.4

In [65]:
team_df.isna().sum()

Team               0
Season             0
Position           0
ø-Age             73
Market value       0
ø-Market value     0
dtype: int64

In [66]:
team_df = team_df.replace('-', np.nan)

In [67]:
team_df.isna().sum()

Team                0
Season              0
Position            0
ø-Age              73
Market value      160
ø-Market value    160
dtype: int64

In [68]:
modern_elo = elo_df_1[elo_df_1['date'].dt.year >= 2004].copy()
modern_elo.drop('change', axis = 1, inplace = True)
modern_elo

,date,team,rating
2049,2004-01-11,United Arab Emirates,1447.0
2048,2004-01-11,Qatar,1519.0
2047,2004-01-11,Oman,1555.0
2051,2004-01-11,Grenada,1251.0
2050,2004-01-11,Barbados,1403.0
...,...,...,...
6673,2025-12-13,Northern Mariana Islands,432.0
6674,2025-12-13,Cocos Islands,422.0
6556,2025-12-13,Azerbaijan,1339.0
6616,2025-12-13,Saint Kitts and Nevis,1005.0


In [69]:
modern_elo.to_csv('elo.csv', index= False)
team_df.to_csv('teamsmtmt.csv', index= False)

In [70]:
elo = pd.read_csv('elo.csv')
teams = pd.read_csv('teamsmtmt.csv')

In [71]:
elo['date'] = pd.DatetimeIndex(elo['date']).year
elo

,date,team,rating
0,2004,United Arab Emirates,1447.0
1,2004,Qatar,1519.0
2,2004,Oman,1555.0
3,2004,Grenada,1251.0
4,2004,Barbados,1403.0
...,...,...,...
4626,2025,Northern Mariana Islands,432.0
4627,2025,Cocos Islands,422.0
4628,2025,Azerbaijan,1339.0
4629,2025,Saint Kitts and Nevis,1005.0


In [72]:
column_to_move = teams.pop('Season')
teams.insert(0, 'date', column_to_move)
teams

,date,Team,Position,ø-Age,Market value,ø-Market value
0,2004,France,Total:,27.8,€466.65m,€9.93m
1,2005,France,Total:,27.3,€333.75m,€10.43m
2,2006,France,Total:,27.3,€350.90m,€10.63m
3,2007,France,Total:,26.4,€357.20m,€9.92m
4,2008,France,Total:,27.9,€544.00m,€13.60m
...,...,...,...,...,...,...
2777,2014,North Macedonia,Total:,26.5,€29.93m,€565k
2778,2025,Oman,Total:,26.9,€12.58m,€257k
2779,2012,Thailand,Total:,28.4,€1.10m,€28k
2780,2022,Trinidad and Tobago,Total:,26.5,€10.73m,€219k


In [73]:
teams.drop('Position', axis = 1, inplace= True)
teams

,date,Team,ø-Age,Market value,ø-Market value
0,2004,France,27.8,€466.65m,€9.93m
1,2005,France,27.3,€333.75m,€10.43m
2,2006,France,27.3,€350.90m,€10.63m
3,2007,France,26.4,€357.20m,€9.92m
4,2008,France,27.9,€544.00m,€13.60m
...,...,...,...,...,...
2777,2014,North Macedonia,26.5,€29.93m,€565k
2778,2025,Oman,26.9,€12.58m,€257k
2779,2012,Thailand,28.4,€1.10m,€28k
2780,2022,Trinidad and Tobago,26.5,€10.73m,€219k


In [74]:
teams.sort_values('date', inplace= True)

In [75]:
teams = teams[teams['date'] < 2026].copy()

In [76]:
teams.to_csv('teamsmtmt.csv', index = False)
elo.to_csv('elo.csv', index= False)

In [77]:
merged_df_2 = pd.merge(elo, teams, how = 'left', left_on = ['date', 'team'], right_on= ['date', 'Team'])
merged_df_2

,date,team,rating,Team,ø-Age,Market value,ø-Market value
0,2004,United Arab Emirates,1447.0,United Arab Emirates,24.0,NaN,NaN
1,2004,Qatar,1519.0,Qatar,23.0,NaN,NaN
2,2004,Oman,1555.0,Oman,21.8,NaN,NaN
3,2004,Grenada,1251.0,NaN,NaN,NaN,NaN
4,2004,Barbados,1403.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
4814,2025,Northern Mariana Islands,432.0,NaN,NaN,NaN,NaN
4815,2025,Cocos Islands,422.0,NaN,NaN,NaN,NaN
4816,2025,Azerbaijan,1339.0,NaN,NaN,NaN,NaN
4817,2025,Saint Kitts and Nevis,1005.0,NaN,NaN,NaN,NaN


In [78]:
elo[elo['team'] == 'Algeria']

,date,team,rating
113,2005,Algeria,1460.0
436,2006,Algeria,1415.0
453,2006,Algeria,1398.0
462,2006,Algeria,1413.0
533,2007,Algeria,1409.0
579,2007,Algeria,1389.0
762,2008,Algeria,1399.0
1191,2011,Algeria,1462.0
1513,2012,Algeria,1500.0
1799,2013,Algeria,1586.0


In [79]:
merged_df_2.drop('Team', axis = 1, inplace= True)
merged_df_2.columns

Index(['date', 'team', 'rating', 'ø-Age', 'Market value', 'ø-Market value'], dtype='str')

In [80]:
merged_df_2['ø-Age'] = merged_df_2.groupby('date')['ø-Age'].transform(lambda x: x.fillna(round(x.mean(), 2)))
merged_df_2

,date,team,rating,ø-Age,Market value,ø-Market value
0,2004,United Arab Emirates,1447.0,24.00,NaN,NaN
1,2004,Qatar,1519.0,23.00,NaN,NaN
2,2004,Oman,1555.0,21.80,NaN,NaN
3,2004,Grenada,1251.0,26.47,NaN,NaN
4,2004,Barbados,1403.0,26.47,NaN,NaN
...,...,...,...,...,...,...
4814,2025,Northern Mariana Islands,432.0,26.30,NaN,NaN
4815,2025,Cocos Islands,422.0,26.30,NaN,NaN
4816,2025,Azerbaijan,1339.0,26.30,NaN,NaN
4817,2025,Saint Kitts and Nevis,1005.0,26.30,NaN,NaN


In [81]:
merged_df_2.isna().sum()

date                 0
team                 0
rating               0
ø-Age                0
Market value      1596
ø-Market value    1596
dtype: int64

In [82]:
merged_df_2['Market value'] = merged_df_2['Market value'].str.replace("€", '')
replacements = {
    'k': 'e3',
    'm': 'e6',
    'bn': 'e9'
}

merged_df_2['Market value'] = merged_df_2['Market value'].replace(replacements, regex = True).astype(float)
merged_df_2

,date,team,rating,ø-Age,Market value,ø-Market value
0,2004,United Arab Emirates,1447.0,24.00,NaN,NaN
1,2004,Qatar,1519.0,23.00,NaN,NaN
2,2004,Oman,1555.0,21.80,NaN,NaN
3,2004,Grenada,1251.0,26.47,NaN,NaN
4,2004,Barbados,1403.0,26.47,NaN,NaN
...,...,...,...,...,...,...
4814,2025,Northern Mariana Islands,432.0,26.30,NaN,NaN
4815,2025,Cocos Islands,422.0,26.30,NaN,NaN
4816,2025,Azerbaijan,1339.0,26.30,NaN,NaN
4817,2025,Saint Kitts and Nevis,1005.0,26.30,NaN,NaN


In [83]:
merged_df_2['ø-Market value'] = merged_df_2['ø-Market value'].str.replace("€", '')

replacements = {
    'k': 'e3',
    'm': 'e6',
    'bn': 'e9'
}

merged_df_2['ø-Market value'] = merged_df_2['ø-Market value'].replace(replacements, regex= True).astype(float)

In [84]:
merged_df_2

,date,team,rating,ø-Age,Market value,ø-Market value
0,2004,United Arab Emirates,1447.0,24.00,NaN,NaN
1,2004,Qatar,1519.0,23.00,NaN,NaN
2,2004,Oman,1555.0,21.80,NaN,NaN
3,2004,Grenada,1251.0,26.47,NaN,NaN
4,2004,Barbados,1403.0,26.47,NaN,NaN
...,...,...,...,...,...,...
4814,2025,Northern Mariana Islands,432.0,26.30,NaN,NaN
4815,2025,Cocos Islands,422.0,26.30,NaN,NaN
4816,2025,Azerbaijan,1339.0,26.30,NaN,NaN
4817,2025,Saint Kitts and Nevis,1005.0,26.30,NaN,NaN


In [85]:
euro_cpi = {
    2004: 81.5,  2005: 83.3,  2006: 85.1,  2007: 86.9,  2008: 89.8,
    2009: 90.1,  2010: 91.5,  2011: 94.0,  2012: 96.3,  2013: 97.6,
    2014: 98.0,  2015: 100.0, 2016: 100.2, 2017: 101.7, 2018: 103.5,
    2019: 104.8, 2020: 105.1, 2021: 107.8, 2022: 116.8, 2023: 123.1,
    2024: 126.0, 2025: 128.5, 2026: 131.0 
}

In [86]:
merged_df_2['cpi_for_year'] = merged_df_2['date'].map(euro_cpi)
cpi_2026 = euro_cpi[2026]
merged_df_2

,date,team,rating,ø-Age,Market value,ø-Market value,cpi_for_year
0,2004,United Arab Emirates,1447.0,24.00,NaN,NaN,81.5
1,2004,Qatar,1519.0,23.00,NaN,NaN,81.5
2,2004,Oman,1555.0,21.80,NaN,NaN,81.5
3,2004,Grenada,1251.0,26.47,NaN,NaN,81.5
4,2004,Barbados,1403.0,26.47,NaN,NaN,81.5
...,...,...,...,...,...,...,...
4814,2025,Northern Mariana Islands,432.0,26.30,NaN,NaN,128.5
4815,2025,Cocos Islands,422.0,26.30,NaN,NaN,128.5
4816,2025,Azerbaijan,1339.0,26.30,NaN,NaN,128.5
4817,2025,Saint Kitts and Nevis,1005.0,26.30,NaN,NaN,128.5


In [87]:
merged_df_2['avg_market_value'] = merged_df_2['ø-Market value'] * (cpi_2026 / merged_df_2['cpi_for_year']).round(3)
merged_df_2

,date,team,rating,ø-Age,Market value,ø-Market value,cpi_for_year,avg_market_value
0,2004,United Arab Emirates,1447.0,24.00,NaN,NaN,81.5,NaN
1,2004,Qatar,1519.0,23.00,NaN,NaN,81.5,NaN
2,2004,Oman,1555.0,21.80,NaN,NaN,81.5,NaN
3,2004,Grenada,1251.0,26.47,NaN,NaN,81.5,NaN
4,2004,Barbados,1403.0,26.47,NaN,NaN,81.5,NaN
...,...,...,...,...,...,...,...,...
4814,2025,Northern Mariana Islands,432.0,26.30,NaN,NaN,128.5,NaN
4815,2025,Cocos Islands,422.0,26.30,NaN,NaN,128.5,NaN
4816,2025,Azerbaijan,1339.0,26.30,NaN,NaN,128.5,NaN
4817,2025,Saint Kitts and Nevis,1005.0,26.30,NaN,NaN,128.5,NaN


In [88]:
merged_df_2 = merged_df_2.drop([ 'cpi_for_year', 'Market value'], axis = 1)
merged_df_2

,date,team,rating,ø-Age,ø-Market value,avg_market_value
0,2004,United Arab Emirates,1447.0,24.00,NaN,NaN
1,2004,Qatar,1519.0,23.00,NaN,NaN
2,2004,Oman,1555.0,21.80,NaN,NaN
3,2004,Grenada,1251.0,26.47,NaN,NaN
4,2004,Barbados,1403.0,26.47,NaN,NaN
...,...,...,...,...,...,...
4814,2025,Northern Mariana Islands,432.0,26.30,NaN,NaN
4815,2025,Cocos Islands,422.0,26.30,NaN,NaN
4816,2025,Azerbaijan,1339.0,26.30,NaN,NaN
4817,2025,Saint Kitts and Nevis,1005.0,26.30,NaN,NaN


In [89]:
merged_df_2['year'] = merged_df_2['date'].astype(int)

In [90]:
merged_df_2['z_score'] = merged_df_2.groupby('year')['ø-Market value'].transform(lambda x: (x - x.mean()) / x.std()).round(2)
merged_df_2

,date,team,rating,ø-Age,ø-Market value,avg_market_value,year,z_score
0,2004,United Arab Emirates,1447.0,24.00,NaN,NaN,2004,NaN
1,2004,Qatar,1519.0,23.00,NaN,NaN,2004,NaN
2,2004,Oman,1555.0,21.80,NaN,NaN,2004,NaN
3,2004,Grenada,1251.0,26.47,NaN,NaN,2004,NaN
4,2004,Barbados,1403.0,26.47,NaN,NaN,2004,NaN
...,...,...,...,...,...,...,...,...
4814,2025,Northern Mariana Islands,432.0,26.30,NaN,NaN,2025,NaN
4815,2025,Cocos Islands,422.0,26.30,NaN,NaN,2025,NaN
4816,2025,Azerbaijan,1339.0,26.30,NaN,NaN,2025,NaN
4817,2025,Saint Kitts and Nevis,1005.0,26.30,NaN,NaN,2025,NaN


In [91]:
merged_df_2[merged_df_2['rating'] <= 1100]

,date,team,rating,ø-Age,ø-Market value,avg_market_value,year,z_score
21,2004,Nicaragua,1013.0,26.47,NaN,NaN,2004,NaN
22,2004,Dominican Republic,1071.0,26.47,NaN,NaN,2004,NaN
78,2004,San Marino,955.0,26.47,NaN,NaN,2004,NaN
82,2004,Andorra,1060.0,26.47,NaN,NaN,2004,NaN
87,2004,Luxembourg,1068.0,25.20,60000.0,96420.0,2004,-0.74
...,...,...,...,...,...,...,...,...
4812,2025,Tonga,520.0,26.30,NaN,NaN,2025,NaN
4813,2025,Niue,496.0,26.30,NaN,NaN,2025,NaN
4814,2025,Northern Mariana Islands,432.0,26.30,NaN,NaN,2025,NaN
4815,2025,Cocos Islands,422.0,26.30,NaN,NaN,2025,NaN


In [92]:
merged_df_2.drop('ø-Market value', axis = 1, inplace= True)

In [93]:
merged_df_2.drop_duplicates()

,date,team,rating,ø-Age,avg_market_value,year,z_score
0,2004,United Arab Emirates,1447.0,24.00,NaN,2004,NaN
1,2004,Qatar,1519.0,23.00,NaN,2004,NaN
2,2004,Oman,1555.0,21.80,NaN,2004,NaN
3,2004,Grenada,1251.0,26.47,NaN,2004,NaN
4,2004,Barbados,1403.0,26.47,NaN,2004,NaN
...,...,...,...,...,...,...,...
4813,2025,Niue,496.0,26.30,NaN,2025,NaN
4814,2025,Northern Mariana Islands,432.0,26.30,NaN,2025,NaN
4815,2025,Cocos Islands,422.0,26.30,NaN,2025,NaN
4816,2025,Azerbaijan,1339.0,26.30,NaN,2025,NaN


In [94]:
ut = set(modern_elo['team'].unique())
ut

{'Afghanistan',
 'Albania',
 'Algeria',
 'Andorra',
 'Angola',
 'Anguilla',
 'Antigua and Barbuda',
 'Argentina',
 'Armenia',
 'Aruba',
 'Australia',
 'Austria',
 'Azerbaijan',
 'Bahamas',
 'Bahrain',
 'Bangladesh',
 'Barbados',
 'Belarus',
 'Belgium',
 'Belize',
 'Benin',
 'Bermuda',
 'Bhutan',
 'Bolivia',
 'Bonaire',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'British Virgin Islands',
 'Brunei',
 'Bulgaria',
 'Burkina Faso',
 'Burundi',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Cape Verde',
 'Cayman Islands',
 'Central African Republic',
 'Chad',
 'Chagos Islands',
 'Chile',
 'China',
 'Christmas Island',
 'Cocos Islands',
 'Colombia',
 'Comoros',
 'Congo',
 'Cook Islands',
 'Costa Rica',
 'Croatia',
 'Cuba',
 'Curaçao',
 'Cyprus',
 'Czechia',
 'Democratic Republic of Congo',
 'Denmark',
 'Djibouti',
 'Dominica',
 'Dominican Republic',
 'East Timor',
 'Eastern Samoa',
 'Ecuador',
 'Egypt',
 'El Salvador',
 'England',
 'Equatorial Guinea',
 'Eritrea',
 'Estonia',
 'Eswatini',
 '

In [95]:
mtut = set((modern_df['home_team'].unique()))
mtut_2 = set(modern_df['away_team'].unique())
mtut.union(mtut_2)
mtut

{'Abkhazia',
 'Afghanistan',
 'Albania',
 'Algeria',
 'Andalusia',
 'Andorra',
 'Angola',
 'Anguilla',
 'Antigua and Barbuda',
 'Argentina',
 'Armenia',
 'Aruba',
 'Australia',
 'Austria',
 'Azerbaijan',
 'Bahamas',
 'Bahrain',
 'Bangladesh',
 'Barawa',
 'Barbados',
 'Basque Country',
 'Belarus',
 'Belgium',
 'Belize',
 'Benin',
 'Bermuda',
 'Bhutan',
 'Bolivia',
 'Bonaire',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'British Virgin Islands',
 'Brittany',
 'Brunei',
 'Bulgaria',
 'Burkina Faso',
 'Burundi',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Canary Islands',
 'Cape Verde',
 'Catalonia',
 'Cayman Islands',
 'Central African Republic',
 'Chad',
 'Chile',
 'China',
 'Colombia',
 'Comoros',
 'Congo',
 'Cook Islands',
 'Corsica',
 'Costa Rica',
 'Croatia',
 'Cuba',
 'Curaçao',
 'Cyprus',
 'Czechia',
 'Denmark',
 'Djibouti',
 'Dominica',
 'Dominican Republic',
 'East Timor',
 'Eastern Samoa',
 'Ecuador',
 'Egypt',
 'El Salvador',
 'England',
 'Equatorial Guinea',
 'Eritrea',
 'E

In [96]:
print(set(unique_teams).issubset(ut))

False


In [97]:
to_drop = set(unique_teams) - ut
print(to_drop)

{0}


In [98]:
set(unique_teams)

{0}

In [99]:
elo_bins = list(range(200, 2350, 50))

merged_df_2['elo_bin'] = pd.cut(merged_df_2['rating'], bins=elo_bins)

merged_df_2['elos'] = merged_df_2.groupby(['year', 'elo_bin'])['z_score'].transform(lambda x: x.fillna(x.mean()))
merged_df_2

,date,team,rating,ø-Age,avg_market_value,year,z_score,elo_bin,elos
0,2004,United Arab Emirates,1447.0,24.00,NaN,2004,NaN,"(1400, 1450]",-0.690000
1,2004,Qatar,1519.0,23.00,NaN,2004,NaN,"(1500, 1550]",-0.595000
2,2004,Oman,1555.0,21.80,NaN,2004,NaN,"(1550, 1600]",-0.570000
3,2004,Grenada,1251.0,26.47,NaN,2004,NaN,"(1250, 1300]",NaN
4,2004,Barbados,1403.0,26.47,NaN,2004,NaN,"(1400, 1450]",-0.690000
...,...,...,...,...,...,...,...,...,...
4814,2025,Northern Mariana Islands,432.0,26.30,NaN,2025,NaN,"(400, 450]",NaN
4815,2025,Cocos Islands,422.0,26.30,NaN,2025,NaN,"(400, 450]",NaN
4816,2025,Azerbaijan,1339.0,26.30,NaN,2025,NaN,"(1300, 1350]",-0.591053
4817,2025,Saint Kitts and Nevis,1005.0,26.30,NaN,2025,NaN,"(1000, 1050]",NaN


In [100]:
merged_df_2[merged_df_2['rating'] <= 1100]

,date,team,rating,ø-Age,avg_market_value,year,z_score,elo_bin,elos
21,2004,Nicaragua,1013.0,26.47,NaN,2004,NaN,"(1000, 1050]",NaN
22,2004,Dominican Republic,1071.0,26.47,NaN,2004,NaN,"(1050, 1100]",-0.74
78,2004,San Marino,955.0,26.47,NaN,2004,NaN,"(950, 1000]",NaN
82,2004,Andorra,1060.0,26.47,NaN,2004,NaN,"(1050, 1100]",-0.74
87,2004,Luxembourg,1068.0,25.20,96420.0,2004,-0.74,"(1050, 1100]",-0.74
...,...,...,...,...,...,...,...,...,...
4812,2025,Tonga,520.0,26.30,NaN,2025,NaN,"(500, 550]",NaN
4813,2025,Niue,496.0,26.30,NaN,2025,NaN,"(450, 500]",NaN
4814,2025,Northern Mariana Islands,432.0,26.30,NaN,2025,NaN,"(400, 450]",NaN
4815,2025,Cocos Islands,422.0,26.30,NaN,2025,NaN,"(400, 450]",NaN


In [101]:
merged_df_2.isna().sum()

date                   0
team                   0
rating                 0
ø-Age                  0
avg_market_value    1596
year                   0
z_score             1596
elo_bin                0
elos                 670
dtype: int64

In [102]:
merged_df_2.to_csv('mdmd.csv')

In [103]:
def extrapolate_minnow_zscore(elo):
    # We set a logical anchor. (Adjust these baseline numbers if needed based on your data)
    if elo >= 1300: return -.73
    elif elo >= 1250: return -.74
    elif elo >= 1200: return -.75
    elif elo >= 1150: return -.76
    elif elo >= 1100: return -.77
    elif elo >= 1050: return -.8
    elif elo >= 1000: return -.83
    elif elo >= 950: return -.88
    elif elo >= 900: return -1
    else: 
        return -2.0

In [104]:
merged_df_2['elos'] = merged_df_2.apply(
    lambda row: extrapolate_minnow_zscore(row['rating']) if pd.isna(row['elos']) else row['elos'], 
    axis=1
)

In [105]:
merged_df_2

,date,team,rating,ø-Age,avg_market_value,year,z_score,elo_bin,elos
0,2004,United Arab Emirates,1447.0,24.00,NaN,2004,NaN,"(1400, 1450]",-0.690000
1,2004,Qatar,1519.0,23.00,NaN,2004,NaN,"(1500, 1550]",-0.595000
2,2004,Oman,1555.0,21.80,NaN,2004,NaN,"(1550, 1600]",-0.570000
3,2004,Grenada,1251.0,26.47,NaN,2004,NaN,"(1250, 1300]",-0.740000
4,2004,Barbados,1403.0,26.47,NaN,2004,NaN,"(1400, 1450]",-0.690000
...,...,...,...,...,...,...,...,...,...
4814,2025,Northern Mariana Islands,432.0,26.30,NaN,2025,NaN,"(400, 450]",-2.000000
4815,2025,Cocos Islands,422.0,26.30,NaN,2025,NaN,"(400, 450]",-2.000000
4816,2025,Azerbaijan,1339.0,26.30,NaN,2025,NaN,"(1300, 1350]",-0.591053
4817,2025,Saint Kitts and Nevis,1005.0,26.30,NaN,2025,NaN,"(1000, 1050]",-0.830000


In [106]:
merged_df_2.drop(['date', 'avg_market_value', 'z_score', 'elo_bin'], axis= 1, inplace= True)
merged_df_2

,team,rating,ø-Age,year,elos
0,United Arab Emirates,1447.0,24.00,2004,-0.690000
1,Qatar,1519.0,23.00,2004,-0.595000
2,Oman,1555.0,21.80,2004,-0.570000
3,Grenada,1251.0,26.47,2004,-0.740000
4,Barbados,1403.0,26.47,2004,-0.690000
...,...,...,...,...,...
4814,Northern Mariana Islands,432.0,26.30,2025,-2.000000
4815,Cocos Islands,422.0,26.30,2025,-2.000000
4816,Azerbaijan,1339.0,26.30,2025,-0.591053
4817,Saint Kitts and Nevis,1005.0,26.30,2025,-0.830000


In [107]:
merged_df_2.isna().sum()

team      0
rating    0
ø-Age     0
year      0
elos      0
dtype: int64

In [108]:
year = merged_df_2.pop('year')
merged_df_2.insert(0, 'Year', year)

In [109]:
merged_df_2.rename(columns= {'elos' : 'market_value_z_score'}, inplace= True)

In [110]:
merged_df_2

,Year,team,rating,ø-Age,market_value_z_score
0,2004,United Arab Emirates,1447.0,24.00,-0.690000
1,2004,Qatar,1519.0,23.00,-0.595000
2,2004,Oman,1555.0,21.80,-0.570000
3,2004,Grenada,1251.0,26.47,-0.740000
4,2004,Barbados,1403.0,26.47,-0.690000
...,...,...,...,...,...
4814,2025,Northern Mariana Islands,432.0,26.30,-2.000000
4815,2025,Cocos Islands,422.0,26.30,-2.000000
4816,2025,Azerbaijan,1339.0,26.30,-0.591053
4817,2025,Saint Kitts and Nevis,1005.0,26.30,-0.830000


In [111]:
merged_df_2.isna().sum()

Year                    0
team                    0
rating                  0
ø-Age                   0
market_value_z_score    0
dtype: int64

In [112]:
modern_df[(modern_df['tournament'] == 'Friendly') & (modern_df['date'].dt.year >= 2018)]

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo
41266,2018-01-07,Estonia,Sweden,1,1,Friendly,1,1495.0,1791.0
41267,2018-01-11,Denmark,Sweden,0,1,Friendly,1,1737.0,1791.0
41268,2018-01-11,Indonesia,Iceland,0,6,Friendly,0,1195.0,1772.0
41269,2018-01-11,Jordan,Finland,1,2,Friendly,1,1484.0,1570.0
41270,2018-01-14,Indonesia,Iceland,1,4,Friendly,0,1195.0,1772.0
...,...,...,...,...,...,...,...,...,...
49206,2026-03-31,Ivory Coast,Scotland,1,0,Friendly,1,1607.0,1790.0
49207,2026-03-31,Jordan,Nigeria,2,2,Friendly,1,1687.0,1627.0
49210,2026-03-31,Mexico,Belgium,1,1,Friendly,1,1835.0,1849.0
49211,2026-03-31,Montenegro,Slovenia,2,3,Friendly,0,1444.0,1695.0


In [113]:
merged_df_2.drop_duplicates(subset= ['Year', 'team'])

,Year,team,rating,ø-Age,market_value_z_score
0,2004,United Arab Emirates,1447.0,24.00,-0.690
1,2004,Qatar,1519.0,23.00,-0.595
2,2004,Oman,1555.0,21.80,-0.570
3,2004,Grenada,1251.0,26.47,-0.740
4,2004,Barbados,1403.0,26.47,-0.690
...,...,...,...,...,...
4811,2025,Kiribati,544.0,26.30,-2.000
4812,2025,Tonga,520.0,26.30,-2.000
4813,2025,Niue,496.0,26.30,-2.000
4814,2025,Northern Mariana Islands,432.0,26.30,-2.000


In [114]:
merged_df_2.to_csv('msg.csv', index = False)
modern_df.to_csv('matches.csv', index= False)

In [115]:
modern_df['date'] = pd.to_datetime(modern_df[('date')]).dt.year
modern_df

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo
27844,2004,Bahrain,Saudi Arabia,0,1,Gulf Cup,1,1479.0,1700.0
27845,2004,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0
27846,2004,Kuwait,Yemen,4,0,Gulf Cup,0,1623.0,1206.0
27847,2004,Oman,Bahrain,0,1,Gulf Cup,1,1252.0,1479.0
27848,2004,United Arab Emirates,Qatar,0,0,Gulf Cup,1,1609.0,1532.0
...,...,...,...,...,...,...,...,...,...
49210,2026,Mexico,Belgium,1,1,Friendly,1,1835.0,1849.0
49211,2026,Montenegro,Slovenia,2,3,Friendly,0,1444.0,1695.0
49212,2026,Morocco,Paraguay,2,1,Friendly,1,1830.0,1833.0
49213,2026,Niger,Togo,0,1,"Morocco, Capital of African Football",1,1403.0,1340.0


In [116]:
final_df = pd.merge(
    modern_df,
    merged_df_2,
    left_on= ['date', 'home_team'],
    right_on= ['Year', 'team'],
    how = 'left'
)

final_df = final_df.rename(columns= {
    'ø-Age' : 'home_team_avg_age',
    'market_value_z_score' : 'home_team_market_value'
})

final_df = pd.merge(
    modern_df,
    merged_df_2,
    left_on= ['date', 'away_team'],
    right_on= ['Year', 'team'],
    how = 'left'
)

final_df = final_df.rename(columns= {
    'ø-Age' : 'away_team_avg_age',
    'market_value_z_score' : 'away_team_market_value'
})#.drop(columns= ['team'])

final_df.drop('Year', axis = 1, inplace= True)

final_df




,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo,team,rating,away_team_avg_age,away_team_market_value
0,2004,Bahrain,Saudi Arabia,0,1,Gulf Cup,1,1479.0,1700.0,NaN,NaN,NaN,NaN
1,2004,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0,Barbados,1403.0,26.47,-0.69
2,2004,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0,Barbados,1338.0,26.47,-0.74
3,2004,Kuwait,Yemen,4,0,Gulf Cup,0,1623.0,1206.0,Yemen,1150.0,26.47,-0.76
4,2004,Oman,Bahrain,0,1,Gulf Cup,1,1252.0,1479.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
31946,2026,Mexico,Belgium,1,1,Friendly,1,1835.0,1849.0,NaN,NaN,NaN,NaN
31947,2026,Montenegro,Slovenia,2,3,Friendly,0,1444.0,1695.0,NaN,NaN,NaN,NaN
31948,2026,Morocco,Paraguay,2,1,Friendly,1,1830.0,1833.0,NaN,NaN,NaN,NaN
31949,2026,Niger,Togo,0,1,"Morocco, Capital of African Football",1,1403.0,1340.0,NaN,NaN,NaN,NaN


In [117]:
final_df = pd.merge(
    modern_df,
    merged_df_2,
    left_on= ['date', 'away_team'],
    right_on= ['Year', 'team'],
    how = 'left'
)

final_df = final_df.rename(columns= {
    'ø-Age' : 'away_team_avg_age',
    'market_value_z_score' : 'away_team_market_value'
})#.drop(columns= ['team'])


In [118]:
final_df

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo,Year,team,rating,away_team_avg_age,away_team_market_value
0,2004,Bahrain,Saudi Arabia,0,1,Gulf Cup,1,1479.0,1700.0,NaN,NaN,NaN,NaN,NaN
1,2004,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0,2004.0,Barbados,1403.0,26.47,-0.69
2,2004,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0,2004.0,Barbados,1338.0,26.47,-0.74
3,2004,Kuwait,Yemen,4,0,Gulf Cup,0,1623.0,1206.0,2004.0,Yemen,1150.0,26.47,-0.76
4,2004,Oman,Bahrain,0,1,Gulf Cup,1,1252.0,1479.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31946,2026,Mexico,Belgium,1,1,Friendly,1,1835.0,1849.0,NaN,NaN,NaN,NaN,NaN
31947,2026,Montenegro,Slovenia,2,3,Friendly,0,1444.0,1695.0,NaN,NaN,NaN,NaN,NaN
31948,2026,Morocco,Paraguay,2,1,Friendly,1,1830.0,1833.0,NaN,NaN,NaN,NaN,NaN
31949,2026,Niger,Togo,0,1,"Morocco, Capital of African Football",1,1403.0,1340.0,NaN,NaN,NaN,NaN,NaN


In [119]:
# final_df.to_csv('final.csv')

In [120]:
unique_teams = set(merged_df_2['team'].unique())

In [121]:
unique_teams

{'Afghanistan',
 'Albania',
 'Algeria',
 'Andorra',
 'Angola',
 'Anguilla',
 'Antigua and Barbuda',
 'Argentina',
 'Armenia',
 'Aruba',
 'Australia',
 'Austria',
 'Azerbaijan',
 'Bahamas',
 'Bahrain',
 'Bangladesh',
 'Barbados',
 'Belarus',
 'Belgium',
 'Belize',
 'Benin',
 'Bermuda',
 'Bhutan',
 'Bolivia',
 'Bonaire',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'British Virgin Islands',
 'Brunei',
 'Bulgaria',
 'Burkina Faso',
 'Burundi',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Cape Verde',
 'Cayman Islands',
 'Central African Republic',
 'Chad',
 'Chagos Islands',
 'Chile',
 'China',
 'Christmas Island',
 'Cocos Islands',
 'Colombia',
 'Comoros',
 'Congo',
 'Cook Islands',
 'Costa Rica',
 'Croatia',
 'Cuba',
 'Curaçao',
 'Cyprus',
 'Czechia',
 'Democratic Republic of Congo',
 'Denmark',
 'Djibouti',
 'Dominica',
 'Dominican Republic',
 'East Timor',
 'Eastern Samoa',
 'Ecuador',
 'Egypt',
 'El Salvador',
 'England',
 'Equatorial Guinea',
 'Eritrea',
 'Estonia',
 'Eswatini',
 '

In [122]:
mtut

{'Abkhazia',
 'Afghanistan',
 'Albania',
 'Algeria',
 'Andalusia',
 'Andorra',
 'Angola',
 'Anguilla',
 'Antigua and Barbuda',
 'Argentina',
 'Armenia',
 'Aruba',
 'Australia',
 'Austria',
 'Azerbaijan',
 'Bahamas',
 'Bahrain',
 'Bangladesh',
 'Barawa',
 'Barbados',
 'Basque Country',
 'Belarus',
 'Belgium',
 'Belize',
 'Benin',
 'Bermuda',
 'Bhutan',
 'Bolivia',
 'Bonaire',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'British Virgin Islands',
 'Brittany',
 'Brunei',
 'Bulgaria',
 'Burkina Faso',
 'Burundi',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Canary Islands',
 'Cape Verde',
 'Catalonia',
 'Cayman Islands',
 'Central African Republic',
 'Chad',
 'Chile',
 'China',
 'Colombia',
 'Comoros',
 'Congo',
 'Cook Islands',
 'Corsica',
 'Costa Rica',
 'Croatia',
 'Cuba',
 'Curaçao',
 'Cyprus',
 'Czechia',
 'Denmark',
 'Djibouti',
 'Dominica',
 'Dominican Republic',
 'East Timor',
 'Eastern Samoa',
 'Ecuador',
 'Egypt',
 'El Salvador',
 'England',
 'Equatorial Guinea',
 'Eritrea',
 'E

In [123]:
teams['date'] = teams['date'].astype(int)

In [124]:
teams.drop_duplicates(subset= ['date', 'Team'])

,date,Team,ø-Age,Market value,ø-Market value
0,2004,France,27.8,€466.65m,€9.93m
1803,2004,Iceland,28.8,€26.90m,€928k
1782,2004,Ghana,24.2,€44.48m,€1.39m
1745,2004,Georgia,26.6,€29.00m,€659k
207,2004,Germany,25.9,NaN,NaN
...,...,...,...,...,...
2037,2025,Gabon,25.6,€37.93m,€824k
1644,2025,Honduras,27.5,€24.20m,€504k
113,2025,Portugal,25.8,€1.12bn,€35.06m
2225,2025,Belarus,25.5,€18.73m,€390k


In [125]:
# final_df = pd.merge(
#     modern_df,
#     teams,
#     left_on= ['date', 'home_team'],
#     right_on= ['date', 'Team'],
#     how = 'left'
# )

# # final_df = final_df.rename(columns= {
# #     'ø-Age' : 'home_team_avg_age',
# #     'Market value' : 'home_team_market_value',
# #     'ø-Market value' : "home_team_avg"
# # })

# final_df = pd.merge(
#     modern_df,
#     teams,
#     left_on= ['date', 'away_team'],
#     right_on= ['date', 'Team'],
#     how = 'left'
# )

# # final_df = final_df.rename(columns= {
# #     'ø-Age' : 'away_team_avg_age',
# #     'Market value' : 'home_team_market_value',
# #     'ø-Market value' : "home_team_avg"
# # })#.drop(columns= ['team'])

# # final_df.drop('Year', axis = 1, inplace= True)

# final_df




In [126]:
# final_df.to_csv('final_1')

In [127]:
merged_df_2 = merged_df_2.drop_duplicates(subset=['Year', 'team'])

In [128]:
final_df = pd.merge(
    modern_df,
    merged_df_2,
    left_on=['date', 'home_team'],
    right_on=['Year', 'team'],
    how='left'
)

final_df = final_df.rename(columns={
    'ø-Age': 'home_team_avg_age',
    'market_value_z_score': 'home_team_market_value'
}).drop(columns=['team', 'Year'])

final_df = pd.merge(
    final_df,
    merged_df_2,
    left_on=['date', 'away_team'],
    right_on=['Year', 'team'],
    how='left'
)

final_df = final_df.rename(columns={
    'ø-Age': 'away_team_avg_age',
    'market_value_z_score': 'away_team_market_value'
}).drop(columns=['team', 'Year'])#, 'rating_y', 'rating_x'])



final_df

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo,rating_x,home_team_avg_age,home_team_market_value,rating_y,away_team_avg_age,away_team_market_value
0,2004,Bahrain,Saudi Arabia,0,1,Gulf Cup,1,1479.0,1700.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2004,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0,1196.0,26.47,-0.76,1403.0,26.47,-0.690
2,2004,Kuwait,Yemen,4,0,Gulf Cup,0,1623.0,1206.0,NaN,NaN,NaN,1150.0,26.47,-0.760
3,2004,Oman,Bahrain,0,1,Gulf Cup,1,1252.0,1479.0,1555.0,21.80,-0.57,NaN,NaN,NaN
4,2004,United Arab Emirates,Qatar,0,0,Gulf Cup,1,1609.0,1532.0,1447.0,24.00,-0.69,1519.0,23.00,-0.595
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20752,2026,Mexico,Belgium,1,1,Friendly,1,1835.0,1849.0,NaN,NaN,NaN,NaN,NaN,NaN
20753,2026,Montenegro,Slovenia,2,3,Friendly,0,1444.0,1695.0,NaN,NaN,NaN,NaN,NaN,NaN
20754,2026,Morocco,Paraguay,2,1,Friendly,1,1830.0,1833.0,NaN,NaN,NaN,NaN,NaN,NaN
20755,2026,Niger,Togo,0,1,"Morocco, Capital of African Football",1,1403.0,1340.0,NaN,NaN,NaN,NaN,NaN,NaN


In [129]:
final_df['home_team_avg_age'] = final_df.groupby('date')['home_team_avg_age'].transform(lambda x: x.fillna(round(x.mean(), 2)))
final_df['away_team_avg_age'] = final_df.groupby('date')['away_team_avg_age'].transform(lambda x: x.fillna(round(x.mean(), 2)))

final_df

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo,rating_x,home_team_avg_age,home_team_market_value,rating_y,away_team_avg_age,away_team_market_value
0,2004,Bahrain,Saudi Arabia,0,1,Gulf Cup,1,1479.0,1700.0,NaN,26.23,NaN,NaN,26.32,NaN
1,2004,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0,1196.0,26.47,-0.76,1403.0,26.47,-0.690
2,2004,Kuwait,Yemen,4,0,Gulf Cup,0,1623.0,1206.0,NaN,26.23,NaN,1150.0,26.47,-0.760
3,2004,Oman,Bahrain,0,1,Gulf Cup,1,1252.0,1479.0,1555.0,21.80,-0.57,NaN,26.32,NaN
4,2004,United Arab Emirates,Qatar,0,0,Gulf Cup,1,1609.0,1532.0,1447.0,24.00,-0.69,1519.0,23.00,-0.595
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20752,2026,Mexico,Belgium,1,1,Friendly,1,1835.0,1849.0,NaN,NaN,NaN,NaN,NaN,NaN
20753,2026,Montenegro,Slovenia,2,3,Friendly,0,1444.0,1695.0,NaN,NaN,NaN,NaN,NaN,NaN
20754,2026,Morocco,Paraguay,2,1,Friendly,1,1830.0,1833.0,NaN,NaN,NaN,NaN,NaN,NaN
20755,2026,Niger,Togo,0,1,"Morocco, Capital of African Football",1,1403.0,1340.0,NaN,NaN,NaN,NaN,NaN,NaN


In [130]:
elo_bins_h = list(range(200, 2350, 50))

final_df['elo_bin_h'] = pd.cut(final_df['home_team_elo'], bins=elo_bins_h)

final_df['home_team_market_value'] = final_df.groupby(['date', 'elo_bin_h'])['home_team_market_value'].transform(lambda x: x.fillna(x.mean())).round(2)

final_df['elo_bin_a'] = pd.cut(final_df['away_team_elo'], bins=elo_bins_h)
final_df['away_team_market_value'] = final_df.groupby(['date', 'elo_bin_h'])['away_team_market_value'].transform(lambda x: x.fillna(x.mean())).round(2)

final_df

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo,rating_x,home_team_avg_age,home_team_market_value,rating_y,away_team_avg_age,away_team_market_value,elo_bin_h,elo_bin_a
0,2004,Bahrain,Saudi Arabia,0,1,Gulf Cup,1,1479.0,1700.0,NaN,26.23,-0.69,NaN,26.32,-0.51,"(1450, 1500]","(1650, 1700]"
1,2004,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0,1196.0,26.47,-0.76,1403.0,26.47,-0.69,"(1300, 1350]","(1300, 1350]"
2,2004,Kuwait,Yemen,4,0,Gulf Cup,0,1623.0,1206.0,NaN,26.23,-0.54,1150.0,26.47,-0.76,"(1600, 1650]","(1200, 1250]"
3,2004,Oman,Bahrain,0,1,Gulf Cup,1,1252.0,1479.0,1555.0,21.80,-0.57,NaN,26.32,-0.66,"(1250, 1300]","(1450, 1500]"
4,2004,United Arab Emirates,Qatar,0,0,Gulf Cup,1,1609.0,1532.0,1447.0,24.00,-0.69,1519.0,23.00,-0.60,"(1600, 1650]","(1500, 1550]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20752,2026,Mexico,Belgium,1,1,Friendly,1,1835.0,1849.0,NaN,NaN,NaN,NaN,NaN,NaN,"(1800, 1850]","(1800, 1850]"
20753,2026,Montenegro,Slovenia,2,3,Friendly,0,1444.0,1695.0,NaN,NaN,NaN,NaN,NaN,NaN,"(1400, 1450]","(1650, 1700]"
20754,2026,Morocco,Paraguay,2,1,Friendly,1,1830.0,1833.0,NaN,NaN,NaN,NaN,NaN,NaN,"(1800, 1850]","(1800, 1850]"
20755,2026,Niger,Togo,0,1,"Morocco, Capital of African Football",1,1403.0,1340.0,NaN,NaN,NaN,NaN,NaN,NaN,"(1400, 1450]","(1300, 1350]"


In [131]:
final_df.isna().sum()

date                         0
home_team                    0
away_team                    0
home_score                   0
away_score                   0
tournament                   0
neutral                      0
home_team_elo                0
away_team_elo                0
rating_x                  6056
home_team_avg_age          165
home_team_market_value     684
rating_y                  6508
away_team_avg_age          165
away_team_market_value     304
elo_bin_h                    0
elo_bin_a                    0
dtype: int64

In [132]:
final_df = final_df[final_df['date'] < 2026].copy()


In [133]:
final_df.isna().sum()

date                         0
home_team                    0
away_team                    0
home_score                   0
away_score                   0
tournament                   0
neutral                      0
home_team_elo                0
away_team_elo                0
rating_x                  5891
home_team_avg_age            0
home_team_market_value     519
rating_y                  6343
away_team_avg_age            0
away_team_market_value     139
elo_bin_h                    0
elo_bin_a                    0
dtype: int64

In [134]:
final_df.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'neutral', 'home_team_elo', 'away_team_elo', 'rating_x',
       'home_team_avg_age', 'home_team_market_value', 'rating_y',
       'away_team_avg_age', 'away_team_market_value', 'elo_bin_h',
       'elo_bin_a'],
      dtype='str')

In [135]:
columns = ['rating_x', 'rating_y', 'elo_bin_h', 'elo_bin_a']

final_df.drop(columns, axis = 1, inplace= True)
final_df

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo,home_team_avg_age,home_team_market_value,away_team_avg_age,away_team_market_value
0,2004,Bahrain,Saudi Arabia,0,1,Gulf Cup,1,1479.0,1700.0,26.23,-0.69,26.32,-0.51
1,2004,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0,26.47,-0.76,26.47,-0.69
2,2004,Kuwait,Yemen,4,0,Gulf Cup,0,1623.0,1206.0,26.23,-0.54,26.47,-0.76
3,2004,Oman,Bahrain,0,1,Gulf Cup,1,1252.0,1479.0,21.80,-0.57,26.32,-0.66
4,2004,United Arab Emirates,Qatar,0,0,Gulf Cup,1,1609.0,1532.0,24.00,-0.69,23.00,-0.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20587,2025,Botswana,Congo,0,3,African Cup of Nations,1,1319.0,1206.0,26.30,-0.59,25.80,-0.26
20588,2025,Equatorial Guinea,Algeria,1,3,African Cup of Nations,1,1437.0,1726.0,25.50,-0.56,26.60,-0.23
20589,2025,Sudan,Burkina Faso,0,2,African Cup of Nations,1,1352.0,1547.0,26.30,-0.57,24.60,-0.41
20590,2025,Gabon,Ivory Coast,2,3,African Cup of Nations,1,1467.0,1607.0,25.60,-0.52,25.80,0.41


In [136]:
final_df.isna().sum()

date                        0
home_team                   0
away_team                   0
home_score                  0
away_score                  0
tournament                  0
neutral                     0
home_team_elo               0
away_team_elo               0
home_team_avg_age           0
home_team_market_value    519
away_team_avg_age           0
away_team_market_value    139
dtype: int64

In [137]:
final_df[final_df['home_team_market_value'].isna()]

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo,home_team_avg_age,home_team_market_value,away_team_avg_age,away_team_market_value
107,2004,United States Virgin Islands,Saint Kitts and Nevis,0,4,FIFA World Cup qualification,0,712.0,1250.0,26.23,NaN,26.47,-0.75
137,2004,Turks and Caicos Islands,Haiti,0,2,FIFA World Cup qualification,1,800.0,1434.0,26.23,NaN,24.40,-0.70
165,2004,Montserrat,Bermuda,0,7,FIFA World Cup qualification,0,606.0,1305.0,26.23,NaN,26.47,-0.76
196,2004,Tajikistan,Bahrain,0,0,FIFA World Cup qualification,0,800.0,1479.0,26.23,NaN,26.32,-0.63
228,2004,Maldives,South Korea,0,0,FIFA World Cup qualification,0,800.0,1693.0,26.23,NaN,26.32,-0.63
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19224,2024,Saint Vincent and the Grenadines,El Salvador,2,3,CONCACAF Nations League,0,1037.0,1483.0,26.70,NaN,26.77,-0.58
19291,2024,Saint Vincent and the Grenadines,El Salvador,2,1,CONCACAF Nations League,0,1037.0,1483.0,26.70,NaN,26.77,-0.58
19292,2024,Montserrat,Bonaire,0,1,CONCACAF Nations League,1,1003.0,861.0,26.70,NaN,26.77,-0.58
19409,2024,Montserrat,Saint Vincent and the Grenadines,1,2,CONCACAF Nations League,1,1003.0,1037.0,26.70,NaN,26.77,-0.58


In [138]:
def extrapolate_minnow_zscore(elo):
    # We set a logical anchor. (Adjust these baseline numbers if needed based on your data)
    if elo >= 1300: return -.73
    elif elo >= 1250: return -.74
    elif elo >= 1200: return -.75
    elif elo >= 1150: return -.76
    elif elo >= 1100: return -.77
    elif elo >= 1050: return -.8
    elif elo >= 1000: return -.83
    elif elo >= 950: return -.88
    elif elo >= 900: return -1
    else: 
        return -2.0

In [139]:
final_df['home_team_market_value'] = final_df.apply(
    lambda row: extrapolate_minnow_zscore(row['home_team_elo']) if pd.isna(row['home_team_market_value']) else row['home_team_market_value'], 
    axis=1
)

final_df['away_team_market_value'] = final_df.apply(
    lambda row: extrapolate_minnow_zscore(row['away_team_elo']) if pd.isna(row['away_team_market_value']) else row['away_team_market_value'], 
    axis=1
)

final_df.isna().sum()

date                      0
home_team                 0
away_team                 0
home_score                0
away_score                0
tournament                0
neutral                   0
home_team_elo             0
away_team_elo             0
home_team_avg_age         0
home_team_market_value    0
away_team_avg_age         0
away_team_market_value    0
dtype: int64

In [140]:
final_df.to_csv('final.csv')

In [141]:
final_df

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo,home_team_avg_age,home_team_market_value,away_team_avg_age,away_team_market_value
0,2004,Bahrain,Saudi Arabia,0,1,Gulf Cup,1,1479.0,1700.0,26.23,-0.69,26.32,-0.51
1,2004,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0,26.47,-0.76,26.47,-0.69
2,2004,Kuwait,Yemen,4,0,Gulf Cup,0,1623.0,1206.0,26.23,-0.54,26.47,-0.76
3,2004,Oman,Bahrain,0,1,Gulf Cup,1,1252.0,1479.0,21.80,-0.57,26.32,-0.66
4,2004,United Arab Emirates,Qatar,0,0,Gulf Cup,1,1609.0,1532.0,24.00,-0.69,23.00,-0.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20587,2025,Botswana,Congo,0,3,African Cup of Nations,1,1319.0,1206.0,26.30,-0.59,25.80,-0.26
20588,2025,Equatorial Guinea,Algeria,1,3,African Cup of Nations,1,1437.0,1726.0,25.50,-0.56,26.60,-0.23
20589,2025,Sudan,Burkina Faso,0,2,African Cup of Nations,1,1352.0,1547.0,26.30,-0.57,24.60,-0.41
20590,2025,Gabon,Ivory Coast,2,3,African Cup of Nations,1,1467.0,1607.0,25.60,-0.52,25.80,0.41


In [142]:
teams

,date,Team,ø-Age,Market value,ø-Market value
0,2004,France,27.8,€466.65m,€9.93m
1803,2004,Iceland,28.8,€26.90m,€928k
1782,2004,Ghana,24.2,€44.48m,€1.39m
1745,2004,Georgia,26.6,€29.00m,€659k
207,2004,Germany,25.9,NaN,NaN
...,...,...,...,...,...
2037,2025,Gabon,25.6,€37.93m,€824k
1644,2025,Honduras,27.5,€24.20m,€504k
113,2025,Portugal,25.8,€1.12bn,€35.06m
2225,2025,Belarus,25.5,€18.73m,€390k


In [143]:
team_df.sort_values('Season')

,Team,Season,Position,ø-Age,Market value,ø-Market value
0,France,2004,Total:,27.8,€466.65m,€9.93m
1803,Iceland,2004,Total:,28.8,€26.90m,€928k
1782,Ghana,2004,Total:,24.2,€44.48m,€1.39m
1745,Georgia,2004,Total:,26.6,€29.00m,€659k
207,Germany,2004,Total:,25.9,NaN,NaN
...,...,...,...,...,...,...
1821,Iceland,2026,Total:,27.7,€81.35m,€3.39m
1125,Venezuela,2026,Total:,25.6,€76.90m,€2.85m
1802,Ghana,2026,Total:,27.0,€230.88m,€8.88m
2564,Mauritania,2026,Total:,26.9,€24.75m,€990k


In [154]:
df_2026 = pd.read_csv('2026_3.csv')


In [145]:
df_2026.to_csv('2026.csv')

In [155]:
unique = list(df_2026['Team'].unique())
unique

['nigeria',
 'australien',
 'algerien',
 'agypten',
 'kanada',
 'norwegen',
 'ukraine',
 'panama',
 'elfenbeinkuste',
 'polen',
 'russland',
 'wales',
 'schweden',
 'serbien',
 'paraguay',
 'tschechien',
 'ungarn',
 'schottland',
 'tunesien',
 'kamerun',
 'demokratische-republik-kongo',
 'griechenland',
 'slowakei',
 'venezuela',
 'usbekistan',
 'costa-rica',
 'mali',
 'peru',
 'chile',
 'katar',
 'rumanien',
 'irak',
 'slowenien',
 'irland',
 'sudafrika',
 'saudi-arabien',
 'burkina-faso',
 'jordanien',
 'albanien',
 'bosnien-herzegowina',
 'honduras',
 'nordmazedonien',
 'vereinigte-arabische-emirate',
 'kap-verde',
 'nordirland',
 'jamaika',
 'georgien',
 'finnland',
 'ghana',
 'island',
 'bolivien',
 'israel',
 'kosovo',
 'oman',
 'guinea',
 'montenegro',
 'curacao',
 'haiti',
 'syrien',
 'neuseeland',
 'bulgarien',
 'gabun',
 'uganda',
 'angola',
 'benin',
 'bahrain',
 'sambia',
 'thailand',
 'china',
 'palastina',
 'guatemala',
 'belarus',
 'luxemburg',
 'vietnam',
 'el-salvador'

In [156]:
len(unique)

175

In [148]:
import json

In [149]:
with open('teams.json', 'r') as file :
    teams_dict = json.load(file)

teams_dict

{'nigeria': '3444',
 'australien': '3433',
 'algerien': '3614',
 'agypten': '3672',
 'kanada': '3510',
 'norwegen': '3440',
 'ukraine': '3699',
 'panama': '3577',
 'elfenbeinkuste': '3591',
 'polen': '3442',
 'russland': '3448',
 'wales': '3864',
 'schweden': '3557',
 'serbien': '3438',
 'paraguay': '3581',
 'tschechien': '3445',
 'ungarn': '3468',
 'schottland': '3380',
 'tunesien': '3670',
 'kamerun': '3434',
 'demokratische-republik-kongo': '3854',
 'griechenland': '3378',
 'slowakei': '3503',
 'venezuela': '3504',
 'usbekistan': '3563',
 'costa-rica': '8497',
 'mali': '3674',
 'peru': '3584',
 'chile': '3700',
 'katar': '14162',
 'rumanien': '3447',
 'irak': '3560',
 'slowenien': '3588',
 'irland': '3509',
 'sudafrika': '3806',
 'saudi-arabien': '3807',
 'burkina-faso': '5872',
 'jordanien': '15737',
 'albanien': '3561',
 'bosnien-herzegowina': '3446',
 'honduras': '3590',
 'nordmazedonien': '5148',
 'vereinigte-arabische-emirate': '5147',
 'kap-verde': '4311',
 'nordirland': '5674

In [150]:
missing_items = {key: teams_dict[key] for key in teams_dict if key not in unique}

In [151]:
missing_items

{'nigeria': '3444',
 'australien': '3433',
 'algerien': '3614',
 'agypten': '3672',
 'kanada': '3510',
 'norwegen': '3440',
 'ukraine': '3699',
 'panama': '3577',
 'elfenbeinkuste': '3591',
 'polen': '3442',
 'russland': '3448',
 'wales': '3864',
 'schweden': '3557',
 'serbien': '3438',
 'paraguay': '3581',
 'tschechien': '3445',
 'ungarn': '3468',
 'schottland': '3380',
 'tunesien': '3670',
 'kamerun': '3434',
 'demokratische-republik-kongo': '3854',
 'griechenland': '3378',
 'slowakei': '3503',
 'venezuela': '3504',
 'usbekistan': '3563',
 'costa-rica': '8497',
 'mali': '3674',
 'peru': '3584',
 'chile': '3700',
 'katar': '14162',
 'rumanien': '3447',
 'irak': '3560',
 'slowenien': '3588',
 'irland': '3509',
 'sudafrika': '3806',
 'saudi-arabien': '3807',
 'burkina-faso': '5872',
 'jordanien': '15737',
 'albanien': '3561',
 'bosnien-herzegowina': '3446',
 'honduras': '3590',
 'nordmazedonien': '5148',
 'vereinigte-arabische-emirate': '5147',
 'kap-verde': '4311',
 'nordirland': '5674

In [152]:
len(teams_dict)

175

In [153]:
{
    "frankreich": "3377",
    "spanien": "3375",
    "argentinien": "3437",
    "england": "3299",
    "portugal": "3300",
    "brasilien": "3439",
    "niederlande": "3379",
    "marokko": "3575",
    "belgien": "3382",
    "deutschland": "3262",
    "kroatien": "3556",
    "italien": "3376",
    "kolumbien": "3816",
    "senegal": "3499",
    "mexiko": "6303",
    "vereinigte-staaten": "3505",
    "uruguay": "3449",
    "japan": "3435",
    "schweiz": "3384",
    "danemark": "3436",
    "iran": "3582",
    "turkei": "3381",
    "ecuador": "5750",
    "osterreich": "3383",
    "sudkorea": "3589"}

{'frankreich': '3377',
 'spanien': '3375',
 'argentinien': '3437',
 'england': '3299',
 'portugal': '3300',
 'brasilien': '3439',
 'niederlande': '3379',
 'marokko': '3575',
 'belgien': '3382',
 'deutschland': '3262',
 'kroatien': '3556',
 'italien': '3376',
 'kolumbien': '3816',
 'senegal': '3499',
 'mexiko': '6303',
 'vereinigte-staaten': '3505',
 'uruguay': '3449',
 'japan': '3435',
 'schweiz': '3384',
 'danemark': '3436',
 'iran': '3582',
 'turkei': '3381',
 'ecuador': '5750',
 'osterreich': '3383',
 'sudkorea': '3589'}

In [157]:
teams_1 = pd.read_csv('2026_2.csv')

wc_df = pd.concat([teams_1, df_2026], ignore_index= True)

wc_df

,Team,Season,Position,ø-Age,Market value,ø-Market value
0,frankreich,2026,Total:,27.0,€1.52bn,€58.58m
1,spanien,2026,Total:,26.7,€1.22bn,€47.03m
2,argentinien,2026,Total:,29.1,€800.50m,€30.79m
3,england,2026,Total:,27.1,€1.36bn,€52.43m
4,portugal,2026,Total:,28.0,€1.01bn,€38.67m
...,...,...,...,...,...,...
195,sao-tome-und-principe,2026,Total:,25.7,€1.43m,€59k
196,dschibuti,2026,Total:,26.3,€575k,€29k
197,somalia,2026,Total:,25.9,€835k,€44k
198,tonga,2026,Total:,23.9,€10k,€0k


In [158]:
wc_df.drop(['Position', 'Market value'], axis= 1, inplace= True)
wc_df

,Team,Season,ø-Age,ø-Market value
0,frankreich,2026,27.0,€58.58m
1,spanien,2026,26.7,€47.03m
2,argentinien,2026,29.1,€30.79m
3,england,2026,27.1,€52.43m
4,portugal,2026,28.0,€38.67m
...,...,...,...,...
195,sao-tome-und-principe,2026,25.7,€59k
196,dschibuti,2026,26.3,€29k
197,somalia,2026,25.9,€44k
198,tonga,2026,23.9,€0k


In [164]:
wc_df = wc_df.replace('-', np.nan)

In [171]:
wc_df.isna().sum()

Team              0
Season            0
ø-Age             0
ø-Market value    0
dtype: int64

In [167]:
wc_df[wc_df['ø-Market value'].isna()]

,Team,Season,ø-Age,ø-Market value
167,papua-neuguinea,2026,29.5,NaN
181,dominica,2026,25.8,NaN
191,amerikanisch-samoa,2026,24.5,NaN
194,cayman-inseln,2026,25.9,NaN


In [170]:
wc_df.dropna(inplace= True)

In [173]:
wc_df['ø-Market value'] = wc_df['ø-Market value'].str.replace("€", '')

replacements = {
    'k': 'e3',
    'm': 'e6',
    'bn': 'e9'
}

wc_df['ø-Market value'] = wc_df['ø-Market value'].replace(replacements, regex= True).astype(float)

wc_df.rename({'ø-Market value' : 'avg_market_value'})

AttributeError: Can only use .str accessor with string values, not floating

In [177]:
wc_df.rename(columns={'ø-Market value' : 'avg_market_value', 'ø-Age' : 'avg_age', 'Season' : 'year'}, inplace= True)
wc_df

,Team,year,avg_age,avg_market_value
0,frankreich,2026,27.0,58580000.0
1,spanien,2026,26.7,47030000.0
2,argentinien,2026,29.1,30790000.0
3,england,2026,27.1,52430000.0
4,portugal,2026,28.0,38670000.0
...,...,...,...,...
195,sao-tome-und-principe,2026,25.7,59000.0
196,dschibuti,2026,26.3,29000.0
197,somalia,2026,25.9,44000.0
198,tonga,2026,23.9,0.0


In [178]:
wc_df['z_score'] = wc_df.groupby('year')['avg_market_value'].transform(lambda x: (x - x.mean()) / x.std()).round(2)

In [179]:
wc_df

,Team,year,avg_age,avg_market_value,z_score
0,frankreich,2026,27.0,58580000.0,5.87
1,spanien,2026,26.7,47030000.0,4.62
2,argentinien,2026,29.1,30790000.0,2.85
3,england,2026,27.1,52430000.0,5.20
4,portugal,2026,28.0,38670000.0,3.71
...,...,...,...,...,...
195,sao-tome-und-principe,2026,25.7,59000.0,-0.48
196,dschibuti,2026,26.3,29000.0,-0.49
197,somalia,2026,25.9,44000.0,-0.48
198,tonga,2026,23.9,0.0,-0.49


In [180]:
wc_teams = ['Australia','Iran', 'Iraq', 'Japan', 'Jordan', 'Qatar', 'Saudi Arabia',
          'South Korea', 'Uzbekistan', 'Algeria', 'Cape Verde', 'Congo', "Ivory Coast",
            'Egypt', 'Ghana', 'Morocco', 'Senegal', 'South Africa','Tunisia', 'Canada',
              'Mexico', 'United States', 'Curaçao', 'Haiti', 'Panama', 'Argentina', 
              'Brazil', 'Colombia', 'Ecuador', 'Paraguay', 'Uruguay', 'New Zealand',
                'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Croatia', 'Czechia'
                  ,'England', 'France' , 'Germany', 'Netherlands', 'Norway',
                    'Portugal', 'Scotland', 'Spain', 'Sweden', 'Switzerland', 'Turkey']
                    

In [182]:
list(wc_df['Team'].unique())

['frankreich',
 'spanien',
 'argentinien',
 'england',
 'portugal',
 'brasilien',
 'niederlande',
 'marokko',
 'belgien',
 'deutschland',
 'kroatien',
 'italien',
 'kolumbien',
 'senegal',
 'mexiko',
 'vereinigte-staaten',
 'uruguay',
 'japan',
 'schweiz',
 'danemark',
 'iran',
 'turkei',
 'ecuador',
 'osterreich',
 'sudkorea',
 'nigeria',
 'australien',
 'algerien',
 'agypten',
 'kanada',
 'norwegen',
 'ukraine',
 'panama',
 'elfenbeinkuste',
 'polen',
 'russland',
 'wales',
 'schweden',
 'serbien',
 'paraguay',
 'tschechien',
 'ungarn',
 'schottland',
 'tunesien',
 'kamerun',
 'demokratische-republik-kongo',
 'griechenland',
 'slowakei',
 'venezuela',
 'usbekistan',
 'costa-rica',
 'mali',
 'peru',
 'chile',
 'katar',
 'rumanien',
 'irak',
 'slowenien',
 'irland',
 'sudafrika',
 'saudi-arabien',
 'burkina-faso',
 'jordanien',
 'albanien',
 'bosnien-herzegowina',
 'honduras',
 'nordmazedonien',
 'vereinigte-arabische-emirate',
 'kap-verde',
 'nordirland',
 'jamaika',
 'georgien',
 'fi

In [183]:
wc_map = {
    "australien": "Australia",
    "iran": "Iran",
    "irak": "Iraq",
    "japan": "Japan",
    "jordanien": "Jordan",
    "katar": "Qatar",
    "saudi-arabien": "Saudi Arabia",
    "sudkorea": "South Korea",
    "usbekistan": "Uzbekistan",
    "algerien": "Algeria",
    "kap-verde": "Cape Verde",
    "republik-kongo": "Congo", 
    "elfenbeinkuste": "Ivory Coast",
    "agypten": "Egypt",
    "ghana": "Ghana",
    "marokko": "Morocco",
    "senegal": "Senegal",
    "sudafrika": "South Africa",
    "tunesien": "Tunisia",
    "kanada": "Canada",
    "mexiko": "Mexico",
    "vereinigte-staaten": "United States",
    "curacao": "Curaçao",
    "haiti": "Haiti",
    "panama": "Panama",
    "argentinien": "Argentina",
    "brasilien": "Brazil",
    "kolumbien": "Colombia",
    "ecuador": "Ecuador",
    "paraguay": "Paraguay",
    "uruguay": "Uruguay",
    "neuseeland": "New Zealand",
    "osterreich": "Austria",
    "belgien": "Belgium",
    "bosnien-herzegowina": "Bosnia and Herzegovina",
    "kroatien": "Croatia",
    "tschechien": "Czechia",
    "england": "England",
    "frankreich": "France",
    "deutschland": "Germany",
    "niederlande": "Netherlands",
    "norwegen": "Norway",
    "portugal": "Portugal",
    "schottland": "Scotland",
    "spanien": "Spain",
    "schweden": "Sweden",
    "schweiz": "Switzerland",
    "turkei": "Turkey"
}

In [184]:
wc_df['Team'] = wc_df['Team'].replace(wc_map)
wc_df

,Team,year,avg_age,avg_market_value,z_score
0,France,2026,27.0,58580000.0,5.87
1,Spain,2026,26.7,47030000.0,4.62
2,Argentina,2026,29.1,30790000.0,2.85
3,England,2026,27.1,52430000.0,5.20
4,Portugal,2026,28.0,38670000.0,3.71
...,...,...,...,...,...
195,sao-tome-und-principe,2026,25.7,59000.0,-0.48
196,dschibuti,2026,26.3,29000.0,-0.49
197,somalia,2026,25.9,44000.0,-0.48
198,tonga,2026,23.9,0.0,-0.49


In [185]:
wc_df = wc_df[wc_df['Team'].isin(wc_teams)]
wc_df

,Team,year,avg_age,avg_market_value,z_score
0,France,2026,27.0,58580000.0,5.87
1,Spain,2026,26.7,47030000.0,4.62
2,Argentina,2026,29.1,30790000.0,2.85
3,England,2026,27.1,52430000.0,5.20
4,Portugal,2026,28.0,38670000.0,3.71
5,Brazil,2026,29.2,35510000.0,3.36
6,Netherlands,2026,27.7,30930000.0,2.87
7,Morocco,2026,26.4,19170000.0,1.59
8,Belgium,2026,27.6,21060000.0,1.80
9,Germany,2026,28.0,37650000.0,3.60


In [194]:
# wc_df.drop('avg_market_value', axis = 1, inplace= True)
wc_df.to_excel('wc_teams.xlsx')

In [189]:
wc_df = pd.read_excel('wc_teams.xlsx')
wc_df

,Team,year,avg_age,z_score,Elo
0,France,2026,27.0,5.87,2062
1,Spain,2026,26.7,4.62,2155
2,Argentina,2026,29.1,2.85,2113
3,England,2026,27.1,5.20,2020
4,Portugal,2026,28.0,3.71,1984
5,Brazil,2026,29.2,3.36,1988
6,Netherlands,2026,27.7,2.87,1944
7,Morocco,2026,26.4,1.59,1824
8,Belgium,2026,27.6,1.80,1888
9,Germany,2026,28.0,3.60,1925


In [192]:
wc_df.sort_values('Elo', ascending= False, inplace= True)

In [193]:
wc_df

,Team,year,avg_age,z_score,Elo
1,Spain,2026,26.7,4.62,2155
2,Argentina,2026,29.1,2.85,2113
0,France,2026,27.0,5.87,2062
3,England,2026,27.1,5.20,2020
5,Brazil,2026,29.2,3.36,1988
4,Portugal,2026,28.0,3.71,1984
11,Colombia,2026,30.1,0.77,1977
6,Netherlands,2026,27.7,2.87,1944
20,Ecuador,2026,26.1,1.05,1935
9,Germany,2026,28.0,3.60,1925


In [196]:
final_df.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'neutral', 'home_team_elo', 'away_team_elo',
       'home_team_avg_age', 'home_team_market_value', 'away_team_avg_age',
       'away_team_market_value'],
      dtype='str')

In [201]:
final_df['is_friendly'] = (final_df['tournament'] == 'Friendly').astype(int)
final_df

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo,home_team_avg_age,home_team_market_value,away_team_avg_age,away_team_market_value,elo_difference,age_difference,value_difference,is_friendly
0,2004,Bahrain,Saudi Arabia,0,1,Gulf Cup,1,1479.0,1700.0,26.23,-0.69,26.32,-0.51,-221.0,-0.09,-0.18,0
1,2004,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0,26.47,-0.76,26.47,-0.69,-34.0,0.00,-0.07,1
2,2004,Kuwait,Yemen,4,0,Gulf Cup,0,1623.0,1206.0,26.23,-0.54,26.47,-0.76,417.0,-0.24,0.22,0
3,2004,Oman,Bahrain,0,1,Gulf Cup,1,1252.0,1479.0,21.80,-0.57,26.32,-0.66,-227.0,-4.52,0.09,0
4,2004,United Arab Emirates,Qatar,0,0,Gulf Cup,1,1609.0,1532.0,24.00,-0.69,23.00,-0.60,77.0,1.00,-0.09,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20587,2025,Botswana,Congo,0,3,African Cup of Nations,1,1319.0,1206.0,26.30,-0.59,25.80,-0.26,113.0,0.50,-0.33,0
20588,2025,Equatorial Guinea,Algeria,1,3,African Cup of Nations,1,1437.0,1726.0,25.50,-0.56,26.60,-0.23,-289.0,-1.10,-0.33,0
20589,2025,Sudan,Burkina Faso,0,2,African Cup of Nations,1,1352.0,1547.0,26.30,-0.57,24.60,-0.41,-195.0,1.70,-0.16,0
20590,2025,Gabon,Ivory Coast,2,3,African Cup of Nations,1,1467.0,1607.0,25.60,-0.52,25.80,0.41,-140.0,-0.20,-0.93,0


In [199]:
final_df['elo_difference'] = (final_df['home_team_elo'] - final_df['away_team_elo']).round(2)
final_df['age_difference'] = (final_df['home_team_avg_age'] - final_df['away_team_avg_age']).round(3)
final_df['value_difference'] = (final_df['home_team_market_value'] - final_df['away_team_market_value']).round(3)

final_df

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_team_elo,away_team_elo,home_team_avg_age,home_team_market_value,away_team_avg_age,away_team_market_value,elo_difference,age_difference,value_difference
0,2004,Bahrain,Saudi Arabia,0,1,Gulf Cup,1,1479.0,1700.0,26.23,-0.69,26.32,-0.51,-221.0,-0.09,-0.18
1,2004,Bermuda,Barbados,0,4,Friendly,0,1305.0,1339.0,26.47,-0.76,26.47,-0.69,-34.0,0.00,-0.07
2,2004,Kuwait,Yemen,4,0,Gulf Cup,0,1623.0,1206.0,26.23,-0.54,26.47,-0.76,417.0,-0.24,0.22
3,2004,Oman,Bahrain,0,1,Gulf Cup,1,1252.0,1479.0,21.80,-0.57,26.32,-0.66,-227.0,-4.52,0.09
4,2004,United Arab Emirates,Qatar,0,0,Gulf Cup,1,1609.0,1532.0,24.00,-0.69,23.00,-0.60,77.0,1.00,-0.09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20587,2025,Botswana,Congo,0,3,African Cup of Nations,1,1319.0,1206.0,26.30,-0.59,25.80,-0.26,113.0,0.50,-0.33
20588,2025,Equatorial Guinea,Algeria,1,3,African Cup of Nations,1,1437.0,1726.0,25.50,-0.56,26.60,-0.23,-289.0,-1.10,-0.33
20589,2025,Sudan,Burkina Faso,0,2,African Cup of Nations,1,1352.0,1547.0,26.30,-0.57,24.60,-0.41,-195.0,1.70,-0.16
20590,2025,Gabon,Ivory Coast,2,3,African Cup of Nations,1,1467.0,1607.0,25.60,-0.52,25.80,0.41,-140.0,-0.20,-0.93


In [202]:
final_df.to_csv('training.csv', index= False)
wc_df.to_csv('prediction.csv', index= False)